# Adapted KoGNER v13 — Fast Iteration with Adaptive Loss & Stability Monitoring

**Google Colab Notebook — A100 GPU Optimized | Self-Contained**

**Reference:** KoGNER (arXiv:2503.15737) — Knowledge-Aware Named Entity Recognition

---

## v13 Changes from v12

| Change | v12 | v13 | Rationale |
|--------|-----|-----|-----------|
| NER loss weight | λ=2.0 | **λ=1.0** | Rebalance NER vs auxiliary losses |
| Sample size | 5000 train | **1500 train** | Fast experimentation |
| Batch size | 16 | **32** | A100 VRAM utilization |
| Adaptive loss | Fixed weights | **Uncertainty-based** | Dynamic rebalancing |
| Loss randomization | None | **Stochastic drop** | Stability analysis |
| Visualization | Basic logging | **Full dashboard** | Live training monitoring |
| GPU target | T4/A100 | **A100 (bf16+TF32)** | Hardware optimization |

### Adaptive Loss (Kendall et al., 2018)
`L_total = Σ (1/(2σ²_i)) · L_i + log(σ_i)` — learns per-task uncertainty σ_i

### Resilience
- Checkpoints saved to **Google Drive** after every epoch
- On disconnect → re-run all cells → resumes from last checkpoint
- Data + knowledge embeddings cached to avoid re-computation

---
## 0. Environment Setup (A100 Optimized)

Install required packages. This notebook is optimized for **A100 GPU** with bf16 and TF32 support.

In [ ]:
# ============================================================================
# ENVIRONMENT SETUP — A100 OPTIMIZED
# ============================================================================
%pip install -q transformers datasets accelerate seqeval faiss-cpu matplotlib scikit-learn seaborn pyyaml

---
## 1. GPU Check, Imports, Drive Mount & Utilities

- A100-specific optimizations: bf16 autocast, TF32 matmul
- Mount Google Drive for persistent checkpoints
- Initialize `ProgressTracker` and `CheckpointManager`

In [ ]:
# ============================================================================
# GPU CHECK & IMPORTS & PROJECT SETUP & UTILITIES
# ============================================================================
import os, sys, json, random, math, time, copy, gc, warnings, pickle
from datetime import datetime
from typing import Dict, List, Tuple, Optional, Any
from collections import Counter, defaultdict
from dataclasses import dataclass, field

import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader, WeightedRandomSampler
from torch.optim import AdamW
from torch.optim.lr_scheduler import OneCycleLR
from transformers import AutoTokenizer, AutoModel, AutoConfig
from datasets import load_dataset
warnings.filterwarnings("ignore")

try:
    import faiss; HAS_FAISS = True; print(f"FAISS: {faiss.__version__}")
except ImportError:
    HAS_FAISS = False; print("WARNING: faiss not installed")

import matplotlib; matplotlib.use("Agg")
import matplotlib.pyplot as plt; import seaborn as sns

SEED = 42
random.seed(SEED); np.random.seed(SEED); torch.manual_seed(SEED)
if torch.cuda.is_available(): torch.cuda.manual_seed_all(SEED)
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
print(f"PyTorch: {torch.__version__}, CUDA: {torch.cuda.is_available()}, Device: {DEVICE}")
if torch.cuda.is_available():
    _props = torch.cuda.get_device_properties(0)
    _vram = getattr(_props, 'total_global_mem', getattr(_props, 'total_mem', 0))
    print(f"GPU: {torch.cuda.get_device_name(0)}, VRAM: {_vram/1e9:.1f} GB")

# ═══════════════════════════════════════════════════════════
# A100 OPTIMIZATIONS
# ═══════════════════════════════════════════════════════════
USE_BF16 = False
USE_TF32 = False
if torch.cuda.is_available():
    gpu_name = torch.cuda.get_device_name(0).lower()
    if "a100" in gpu_name or "h100" in gpu_name:
        USE_BF16 = True
        USE_TF32 = True
        torch.backends.cuda.matmul.allow_tf32 = True
        torch.backends.cudnn.allow_tf32 = True
        print("A100/H100 detected: bf16 autocast + TF32 matmul ENABLED")
    else:
        print(f"GPU: {gpu_name} — using fp32 (bf16/TF32 disabled)")

# ═══════════════════════════════════════════════════════════
# GOOGLE DRIVE MOUNT
# ═══════════════════════════════════════════════════════════
DRIVE_BASE = "/content/drive/MyDrive/NER_v13"
LOCAL_BASE = "/tmp/results_v13"

try:
    from google.colab import drive
    drive.mount("/content/drive", force_remount=False)
    os.makedirs(DRIVE_BASE, exist_ok=True)
    DRIVE_AVAILABLE = True
    print(f"Google Drive mounted: {DRIVE_BASE}")
except Exception as e:
    DRIVE_AVAILABLE = False
    print(f"Drive not available ({e}), using local storage only")

os.makedirs(LOCAL_BASE, exist_ok=True)

# ═══════════════════════════════════════════════════════════
# TRAINING HISTORY — stores all metrics for visualization
# ═══════════════════════════════════════════════════════════
class TrainingHistory:
    """Tracks all training metrics per epoch for visualization."""
    def __init__(self):
        self.epochs = []
        self.train_loss = []
        self.val_loss = []
        self.overall_f1 = []
        self.minority_f1 = []
        self.gene_f1 = []
        self.disease_f1 = []
        self.chemical_f1 = []
        self.loss_components = []  # list of dicts per epoch
        self.gradient_norms = []   # list of dicts per epoch
        self.adaptive_weights = [] # list of dicts per epoch (uncertainty σ)
        self.stages = []           # stage name per epoch
        self.lr_history = []       # learning rate per epoch

    def record_epoch(self, epoch, stage, train_loss, val_metrics,
                     loss_comps=None, grad_norms=None, adaptive_wts=None, lr=None):
        self.epochs.append(epoch)
        self.stages.append(stage)
        self.train_loss.append(train_loss)
        self.overall_f1.append(val_metrics.get("overall_micro_f1", 0))
        self.minority_f1.append(val_metrics.get("minority_micro_f1", 0))
        self.gene_f1.append(val_metrics.get("gene_f1", 0))
        self.disease_f1.append(val_metrics.get("disease_f1", 0))
        self.chemical_f1.append(val_metrics.get("chemical_f1", 0))
        self.loss_components.append(loss_comps or {})
        self.gradient_norms.append(grad_norms or {})
        self.adaptive_weights.append(adaptive_wts or {})
        self.lr_history.append(lr or 0)

    def to_dict(self):
        return {k: v for k, v in self.__dict__.items()}

    @classmethod
    def from_dict(cls, d):
        h = cls()
        for k, v in d.items():
            if hasattr(h, k):
                setattr(h, k, v)
        return h

HISTORY = TrainingHistory()

# ═══════════════════════════════════════════════════════════
# PROGRESS TRACKER & LOGGING
# ═══════════════════════════════════════════════════════════
STATUS_FILE = "/tmp/ner_v13_monitor.json"
LOG_FILE = "/tmp/v13_training.log"

class ProgressTracker:
    def __init__(self):
        self.data = {"version": "v13", "pipeline_start": time.time(),
            "current_phase": "INIT", "phases_completed": [], "gpu": {},
            "training": {}, "ablation": {}, "results": {}, "errors": [],
            "last_update": time.time(), "elapsed_min": 0.0}
        self._save()
    def _save(self):
        self.data["last_update"] = time.time()
        self.data["elapsed_min"] = round((time.time() - self.data["pipeline_start"]) / 60, 1)
        if torch.cuda.is_available():
            self.data["gpu"] = {"alloc_gb": round(torch.cuda.memory_allocated()/1e9, 2),
                "reserved_gb": round(torch.cuda.memory_reserved()/1e9, 2)}
        try:
            with open(STATUS_FILE, "w") as f: json.dump(self.data, f, indent=1)
        except: pass
    def set_phase(self, p): self.data["current_phase"] = p; self._save()
    def complete_phase(self, p):
        if p not in self.data["phases_completed"]: self.data["phases_completed"].append(p)
        self._save()
    def update_training(self, **kw): self.data["training"].update(kw); self._save()
    def update_ablation(self, **kw): self.data["ablation"].update(kw); self._save()
    def update_results(self, k, v): self.data["results"][k] = v; self._save()
    def log_error(self, e): self.data["errors"].append({"t": time.time(), "e": str(e)}); self._save()
    def set_complete(self): self.data["current_phase"] = "COMPLETE"; self._save()

PROGRESS = ProgressTracker()

def log(msg):
    ts = datetime.now().strftime("%H:%M:%S")
    line = f"[{ts}] {msg}"
    print(line, flush=True)
    try:
        with open(LOG_FILE, "a") as f: f.write(line + "\n")
    except: pass

# ═══════════════════════════════════════════════════════════
# CHECKPOINT MANAGER
# ═══════════════════════════════════════════════════════════
class CheckpointManager:
    def __init__(self, base_dir=DRIVE_BASE, local_dir=LOCAL_BASE):
        self.base_dir = base_dir if DRIVE_AVAILABLE else local_dir
        self.local_dir = local_dir
        self.ckpt_dir = os.path.join(self.base_dir, "checkpoints")
        os.makedirs(self.ckpt_dir, exist_ok=True)
        log(f"CheckpointManager: {self.ckpt_dir}")
    def _path(self, name): return os.path.join(self.ckpt_dir, name)
    def save_data(self, data, train_knowledge, val_knowledge, test_knowledge):
        path = self._path("data_cache.pkl")
        with open(path, "wb") as f:
            pickle.dump({"data": data, "train_k": train_knowledge,
                         "val_k": val_knowledge, "test_k": test_knowledge}, f)
        log(f"Data cached to {path}")
    def load_data(self):
        path = self._path("data_cache.pkl")
        if os.path.exists(path):
            with open(path, "rb") as f: cache = pickle.load(f)
            log(f"Data loaded from cache: {path}")
            return cache["data"], cache["train_k"], cache["val_k"], cache["test_k"]
        return None
    def save_training_checkpoint(self, experiment_name, epoch, model, optimizer,
                                  scheduler, best_f1, best_state, history_dict):
        path = self._path(f"train_{experiment_name}.pt")
        ckpt = {"experiment_name": experiment_name, "epoch": epoch,
            "model_state_dict": model.state_dict(),
            "optimizer_state_dict": optimizer.state_dict(),
            "scheduler_state_dict": scheduler.state_dict(),
            "best_f1": best_f1, "best_state": best_state,
            "history": history_dict}
        torch.save(ckpt, path)
        log(f"Checkpoint: {experiment_name} epoch {epoch+1}")
    def load_training_checkpoint(self, experiment_name):
        path = self._path(f"train_{experiment_name}.pt")
        if os.path.exists(path):
            ckpt = torch.load(path, map_location="cpu", weights_only=False)
            log(f"Checkpoint loaded: {experiment_name} epoch {ckpt['epoch']+1}")
            return ckpt
        return None
    def save_ablation_state(self, completed_ablations, all_results):
        path = self._path("ablation_state.json")
        state = {"completed": list(completed_ablations), "results": {}}
        for name, res in all_results.items():
            state["results"][name] = res if "error" not in res else {"error": res["error"]}
        with open(path, "w") as f: json.dump(state, f, indent=2, default=str)
    def load_ablation_state(self):
        path = self._path("ablation_state.json")
        if os.path.exists(path):
            with open(path) as f: return json.load(f)
        return None
    def save_best_model(self, experiment_name, model_state, metrics):
        torch.save({"model_state_dict": model_state, "metrics": metrics},
                    self._path(f"best_{experiment_name}.pt"))
        with open(self._path(f"metrics_{experiment_name}.json"), "w") as f:
            json.dump(metrics, f, indent=2)
    def save_final_results(self, all_results):
        with open(os.path.join(self.base_dir, "metrics_v13.json"), "w") as f:
            json.dump(all_results, f, indent=2)
    def has_completed_experiment(self, name):
        return os.path.exists(self._path(f"best_{name}.pt"))
    def save_history(self, history):
        path = self._path("training_history.json")
        with open(path, "w") as f: json.dump(history.to_dict(), f, indent=2)
    def load_history(self):
        path = self._path("training_history.json")
        if os.path.exists(path):
            with open(path) as f: return TrainingHistory.from_dict(json.load(f))
        return None

CKPT = CheckpointManager()
log("v13 Pipeline initialized — A100 optimized, adaptive loss, visualization")

---
## 1b. Model Components (Inlined — Self-Contained)

All model components from `src/` are inlined below so this notebook runs **without** the project directory.

Components defined in the following cells:
- **Encoders**: `KnowledgeAwareEncoder` [1], `KnowledgeEncoder` [2], `KnowledgeFusionLayer` [3]
- **Spans**: `SpanRepresentationLayer` [4], `SpanPruner` [4b], `EntityTypeEncoder` [5]
- **Matching**: `SpanEntityMatcherV11_1` [6], `SpanEvidenceAggregator` [6b], `GatedSpanTokenFusion` [9]
- **Decoder**: `CRFDecoder` [10]
- **KG Teacher**: `GraphTransformer`, `TransR`, `KGTeacher` [7]
- **Losses**: `FocalLoss`, `BoundaryLoss`, `EntityTypeLoss`, `KnowledgeAlignmentLoss`, `ContrastiveEntityLoss`, `KGDistillationLoss`, `CompositeLossV12`
- **Schedule**: `TrainingSchedule` (3-stage)
- **Main Model**: `KoGNERv12` (`KnowledgeAwareNERModelV11_6`)

In [ ]:
# ============================================================================
# [1] KnowledgeAwareEncoder — BioBERT backbone
# [2] KnowledgeEncoder — FAISS retrieval projection
# [3] KnowledgeFusionLayer — concat | gated | attention
# ============================================================================

# ── [1] BioBERT Text Encoder ──────────────────────────────────────────────
class KnowledgeAwareEncoder(nn.Module):
    """BioBERT-based encoder producing token-level hidden states."""
    def __init__(self, encoder_name="dmis-lab/biobert-base-cased-v1.2",
                 hidden_dim=768, dropout=0.1, freeze_encoder=False):
        super().__init__()
        self.encoder = AutoModel.from_pretrained(encoder_name)
        self.hidden_dim = hidden_dim
        self.dropout = nn.Dropout(dropout)
        if freeze_encoder:
            for param in self.encoder.parameters():
                param.requires_grad = False
        encoder_hidden = self.encoder.config.hidden_size
        self.needs_projection = encoder_hidden != hidden_dim
        if self.needs_projection:
            self.projection = nn.Linear(encoder_hidden, hidden_dim)

    def forward(self, input_ids, attention_mask, token_type_ids=None):
        kwargs = {"input_ids": input_ids, "attention_mask": attention_mask}
        if token_type_ids is not None:
            kwargs["token_type_ids"] = token_type_ids
        outputs = self.encoder(**kwargs)
        token_repr = outputs.last_hidden_state
        if self.needs_projection:
            token_repr = self.projection(token_repr)
        return self.dropout(token_repr)

    def get_hidden_dim(self): return self.hidden_dim


# ── [2] Knowledge Encoder ─────────────────────────────────────────────────
class KnowledgeEncoder(nn.Module):
    """Encodes FAISS-retrieved knowledge embeddings for fusion."""
    def __init__(self, embedding_dim=768, hidden_dim=768, dropout=0.1,
                 top_k=5, mode="broadcast"):
        super().__init__()
        self.embedding_dim = embedding_dim
        self.hidden_dim = hidden_dim
        self.top_k = top_k
        self.mode = mode
        self.knowledge_proj = nn.Sequential(
            nn.Linear(embedding_dim, hidden_dim), nn.GELU(),
            nn.Dropout(dropout), nn.Linear(hidden_dim, hidden_dim))
        self.attention_pool = nn.Sequential(nn.Linear(hidden_dim, 1))
        self.layer_norm = nn.LayerNorm(hidden_dim)
        self.dropout = nn.Dropout(dropout)

    def forward(self, knowledge_embeddings, seq_len, attention_mask=None):
        projected = self.knowledge_proj(knowledge_embeddings)
        if self.mode == "token_aware":
            return self.dropout(self.layer_norm(projected))
        attn_scores = self.attention_pool(projected).squeeze(-1)
        attn_weights = torch.softmax(attn_scores, dim=-1)
        pooled = torch.bmm(attn_weights.unsqueeze(1), projected).squeeze(1)
        pooled = self.dropout(self.layer_norm(pooled))
        knowledge_repr = pooled.unsqueeze(1).expand(-1, seq_len, -1)
        if attention_mask is not None:
            knowledge_repr = knowledge_repr * attention_mask.unsqueeze(-1).float()
        return knowledge_repr


# ── [3] Knowledge Fusion Layer ────────────────────────────────────────────
class ConcatFusion(nn.Module):
    def __init__(self, hidden_dim=768, dropout=0.1):
        super().__init__()
        self.projection = nn.Sequential(
            nn.Linear(hidden_dim * 2, hidden_dim), nn.GELU(), nn.Dropout(dropout))
        self.layer_norm = nn.LayerNorm(hidden_dim)
    def forward(self, token_repr, knowledge_repr, knowledge_mask=None):
        combined = torch.cat([token_repr, knowledge_repr], dim=-1)
        fused = self.projection(combined)
        return self.layer_norm(fused + token_repr)


class GatedFusion(nn.Module):
    def __init__(self, hidden_dim=768, gate_hidden_dim=256, dropout=0.1):
        super().__init__()
        self.gate_network = nn.Sequential(
            nn.Linear(hidden_dim * 2, gate_hidden_dim), nn.ReLU(),
            nn.Dropout(dropout), nn.Linear(gate_hidden_dim, hidden_dim), nn.Sigmoid())
        self.layer_norm = nn.LayerNorm(hidden_dim)
        self.dropout = nn.Dropout(dropout)
    def forward(self, token_repr, knowledge_repr, knowledge_mask=None):
        combined = torch.cat([token_repr, knowledge_repr], dim=-1)
        gate = self.gate_network(combined)
        fused = gate * token_repr + (1 - gate) * knowledge_repr
        return self.dropout(self.layer_norm(fused + token_repr))


class AttentionFusion(nn.Module):
    def __init__(self, hidden_dim=768, num_heads=4, dropout=0.1,
                 use_gated_residual=True, residual_gate_dim=256):
        super().__init__()
        self.hidden_dim = hidden_dim
        self.num_heads = num_heads
        self.head_dim = hidden_dim // num_heads
        assert hidden_dim % num_heads == 0
        self.q_proj = nn.Linear(hidden_dim, hidden_dim)
        self.k_proj = nn.Linear(hidden_dim, hidden_dim)
        self.v_proj = nn.Linear(hidden_dim, hidden_dim)
        self.out_proj = nn.Linear(hidden_dim, hidden_dim)
        self.attn_dropout = nn.Dropout(dropout)
        self.out_dropout = nn.Dropout(dropout)
        self.layer_norm = nn.LayerNorm(hidden_dim)
        self.use_gated_residual = use_gated_residual
        if use_gated_residual:
            self.residual_gate = nn.Sequential(
                nn.Linear(hidden_dim * 2, residual_gate_dim), nn.ReLU(),
                nn.Dropout(dropout), nn.Linear(residual_gate_dim, hidden_dim), nn.Sigmoid())

    def forward(self, token_repr, knowledge_repr, knowledge_mask=None):
        batch_size, seq_len, _ = token_repr.shape
        kv_len = knowledge_repr.size(1)
        Q = self.q_proj(token_repr).view(batch_size, seq_len, self.num_heads, self.head_dim).transpose(1, 2)
        K = self.k_proj(knowledge_repr).view(batch_size, kv_len, self.num_heads, self.head_dim).transpose(1, 2)
        V = self.v_proj(knowledge_repr).view(batch_size, kv_len, self.num_heads, self.head_dim).transpose(1, 2)
        scale = math.sqrt(self.head_dim)
        attn_scores = torch.matmul(Q, K.transpose(-2, -1)) / scale
        if knowledge_mask is not None:
            attn_mask = knowledge_mask.unsqueeze(1).unsqueeze(2)
            attn_scores = attn_scores.masked_fill(attn_mask == 0, float("-inf"))
        attn_weights = F.softmax(attn_scores, dim=-1)
        attn_weights = self.attn_dropout(attn_weights)
        attn_output = torch.matmul(attn_weights, V)
        attn_output = attn_output.transpose(1, 2).contiguous().view(batch_size, seq_len, self.hidden_dim)
        attn_output = self.out_dropout(self.out_proj(attn_output))
        if self.use_gated_residual:
            combined = torch.cat([token_repr, attn_output], dim=-1)
            gate = self.residual_gate(combined)
            fused = gate * token_repr + (1 - gate) * attn_output
        else:
            fused = token_repr + attn_output
        return self.layer_norm(fused)


class KnowledgeFusionLayer(nn.Module):
    """Unified knowledge fusion: concat / gated / attention / none."""
    def __init__(self, config: dict):
        super().__init__()
        method = config.get("method", "attention")
        hidden_dim = config.get("hidden_dim", 768)
        if method == "concat":
            cfg = config.get("concat", {})
            self.fusion = ConcatFusion(hidden_dim, cfg.get("dropout", 0.1))
        elif method == "gated":
            cfg = config.get("gated", {})
            self.fusion = GatedFusion(hidden_dim, cfg.get("gate_hidden_dim", 256), cfg.get("dropout", 0.1))
        elif method == "attention":
            cfg = config.get("attention", {})
            self.fusion = AttentionFusion(hidden_dim, cfg.get("num_heads", 4), cfg.get("dropout", 0.1),
                cfg.get("use_gated_residual", True), cfg.get("residual_gate_dim", 256))
        elif method == "none":
            self.fusion = None
        else:
            raise ValueError(f"Unknown fusion method: {method}")
        self.method = method

    def forward(self, token_repr, knowledge_repr, knowledge_mask=None):
        if self.fusion is None:
            return token_repr
        return self.fusion(token_repr, knowledge_repr, knowledge_mask)

log("Loaded: KnowledgeAwareEncoder, KnowledgeEncoder, KnowledgeFusionLayer")

In [ ]:
# ============================================================================
# [4] SpanRepresentationLayer — candidate span encoding
# [4b] SpanPruner — top-k span selection
# [5] EntityTypeEncoder — bi-encoder entity type arm
# ============================================================================

class SpanRepresentationLayer(nn.Module):
    """Creates span representations from token-level hidden states."""
    def __init__(self, hidden_dim=768, span_ffn_dim=1024, max_span_width=8,
                 use_width_embedding=True, width_embedding_dim=150,
                 dropout=0.1, pooling="endpoint"):
        super().__init__()
        self.hidden_dim = hidden_dim
        self.max_span_width = max_span_width
        self.pooling = pooling
        self.use_width_embedding = use_width_embedding
        if use_width_embedding:
            self.width_embedding = nn.Embedding(max_span_width, width_embedding_dim)
            input_dim = self._compute_input_dim(hidden_dim, width_embedding_dim)
        else:
            input_dim = self._compute_input_dim(hidden_dim, 0)
        self.span_ffn = nn.Sequential(
            nn.Linear(input_dim, span_ffn_dim), nn.GELU(),
            nn.Dropout(dropout), nn.Linear(span_ffn_dim, hidden_dim))
        self.layer_norm = nn.LayerNorm(hidden_dim)
        self.dropout = nn.Dropout(dropout)
        if pooling == "attention":
            self.attn_score = nn.Sequential(nn.Linear(hidden_dim, 1))

    def _compute_input_dim(self, hidden_dim, width_dim):
        if self.pooling == "endpoint":
            return hidden_dim * 2 + width_dim
        return hidden_dim + width_dim

    def forward(self, token_repr, attention_mask=None):
        batch_size, seq_len, _ = token_repr.shape
        device = token_repr.device
        span_indices = []
        for start in range(seq_len):
            for width in range(1, self.max_span_width + 1):
                end = start + width - 1
                if end < seq_len:
                    span_indices.append((start, end))
        num_spans = len(span_indices)
        if num_spans == 0:
            return (torch.zeros(batch_size, 0, self.hidden_dim, device=device),
                    torch.zeros(batch_size, 0, device=device), [])
        start_indices = torch.tensor([s for s, e in span_indices], device=device)
        end_indices = torch.tensor([e for s, e in span_indices], device=device)
        widths = end_indices - start_indices
        start_repr = token_repr[:, start_indices, :]
        end_repr = token_repr[:, end_indices, :]
        if self.pooling == "endpoint":
            span_features = torch.cat([start_repr, end_repr], dim=-1)
        elif self.pooling in ("maxpool", "meanpool", "attention"):
            token_ids, span_tok_mask = self._build_span_token_ids(start_indices, widths, device)
            span_tokens = token_repr[:, token_ids, :]
            if self.pooling == "maxpool":
                mask = span_tok_mask.unsqueeze(0).unsqueeze(-1)
                span_tokens_m = span_tokens.masked_fill(~mask, float("-inf"))
                span_features, _ = span_tokens_m.max(dim=2)
            elif self.pooling == "meanpool":
                mask = span_tok_mask.unsqueeze(0).unsqueeze(-1).float()
                span_features = (span_tokens * mask).sum(dim=2) / mask.sum(dim=2).clamp(min=1.0)
            else:
                B, S, W, H = span_tokens.shape
                scores = self.attn_score(span_tokens).squeeze(-1)
                m = span_tok_mask.unsqueeze(0).expand(B, -1, -1)
                scores = scores.masked_fill(~m, float("-inf"))
                weights = F.softmax(scores, dim=-1).unsqueeze(-1)
                weights = weights.masked_fill(~m.unsqueeze(-1), 0.0)
                span_features = (span_tokens * weights).sum(dim=2)
        else:
            raise ValueError(f"Unknown pooling: {self.pooling}")
        if self.use_width_embedding:
            width_emb = self.width_embedding(widths).unsqueeze(0).expand(batch_size, -1, -1)
            span_features = torch.cat([span_features, width_emb], dim=-1)
        span_repr = self.dropout(self.layer_norm(self.span_ffn(span_features)))
        if attention_mask is not None:
            start_valid = attention_mask[:, start_indices]
            end_valid = attention_mask[:, end_indices]
            span_mask = (start_valid * end_valid).float()
        else:
            span_mask = torch.ones(batch_size, num_spans, device=device)
        return span_repr, span_mask, span_indices

    def _build_span_token_ids(self, start_indices, widths, device):
        max_w = self.max_span_width
        offsets = torch.arange(max_w, device=device).unsqueeze(0)
        token_ids = start_indices.unsqueeze(1) + offsets
        span_tok_mask = offsets <= widths.unsqueeze(1)
        token_ids = token_ids.clamp(max=token_ids.max().item()) * span_tok_mask.long()
        return token_ids, span_tok_mask

    def get_span_labels(self, ner_labels, span_indices):
        batch_size = ner_labels.size(0)
        device = ner_labels.device
        num_spans = len(span_indices)
        if num_spans == 0:
            return torch.zeros(batch_size, 0, device=device, dtype=torch.long)
        starts = torch.tensor([s for s, e in span_indices], device=device)
        ends = torch.tensor([e for s, e in span_indices], device=device)
        widths = ends - starts
        start_labels = ner_labels[:, starts]
        is_b_tag = (start_labels == 1) | (start_labels == 3) | (start_labels == 5)
        is_single = widths.unsqueeze(0) == 0
        span_labels = (is_b_tag & is_single).long()
        multi_mask = is_b_tag & ~is_single
        if multi_mask.any():
            expected_i = start_labels + 1
            max_w = self.max_span_width
            offsets = torch.arange(1, max_w, device=device).unsqueeze(0)
            cont_ids = starts.unsqueeze(1) + offsets
            cont_valid = offsets <= widths.unsqueeze(1)
            cont_ids = cont_ids.clamp(max=ner_labels.size(1) - 1)
            cont_labels = ner_labels[:, cont_ids.reshape(-1)].reshape(batch_size, num_spans, max_w - 1)
            expected_expanded = expected_i.unsqueeze(2).expand_as(cont_labels)
            matches = (cont_labels == expected_expanded) | ~cont_valid.unsqueeze(0)
            all_match = matches.all(dim=2)
            span_labels = span_labels | (multi_mask & all_match).long()
        return span_labels


# ── [4b] Span Pruner ─────────────────────────────────────────────────────
class SpanPruner(nn.Module):
    """Scores candidate spans and retains top-k per sequence."""
    def __init__(self, hidden_dim=768, max_spans_per_sequence=100, dropout=0.1):
        super().__init__()
        self.max_spans = max_spans_per_sequence
        self.span_scorer = nn.Sequential(
            nn.Linear(hidden_dim, hidden_dim // 4), nn.GELU(),
            nn.Dropout(dropout), nn.Linear(hidden_dim // 4, 1))

    def forward(self, span_repr, span_mask, span_indices):
        batch_size, num_spans, hidden_dim = span_repr.shape
        scores = self.span_scorer(span_repr).squeeze(-1)
        scores = scores * span_mask + (-1e9) * (1 - span_mask)
        k = min(self.max_spans, num_spans)
        top_scores, top_indices = torch.topk(scores, k=k, dim=1)
        gather_idx = top_indices.unsqueeze(-1).expand(-1, -1, hidden_dim)
        pruned_repr = torch.gather(span_repr, 1, gather_idx)
        pruned_mask = torch.gather(span_mask, 1, top_indices)
        top_idx_first = top_indices[0].cpu().tolist()
        pruned_indices = [span_indices[i] for i in top_idx_first]
        return pruned_repr, pruned_mask, pruned_indices, scores, top_indices


# ── [5] Entity Type Encoder ──────────────────────────────────────────────
DEFAULT_ENTITY_TYPE_DESCRIPTIONS = {
    "GENE": "gene protein DNA RNA enzyme kinase receptor transcription factor molecular biology",
    "DISEASE": "disease disorder syndrome illness condition pathology medical diagnosis clinical",
    "CHEMICAL": "chemical drug compound molecule medication pharmaceutical substance treatment therapy",
}

class EntityTypeEncoder(nn.Module):
    """Encodes entity type labels into dense representations for bi-encoder matching."""
    def __init__(self, entity_type_descriptions, hidden_dim=768,
                 encoder_name="dmis-lab/biobert-base-cased-v1.2",
                 use_pretrained_encoder=True, freeze_type_encoder=True,
                 ffn_dim=1024, dropout=0.1):
        super().__init__()
        self.hidden_dim = hidden_dim
        self.entity_types = list(entity_type_descriptions.keys())
        self.type_descriptions = entity_type_descriptions
        self.num_types = len(self.entity_types)
        self.type_to_idx = {t: i for i, t in enumerate(self.entity_types)}
        if use_pretrained_encoder:
            self._init_pretrained_embeddings(encoder_name)
        else:
            self.type_embeddings = nn.Embedding(self.num_types, hidden_dim)
        self.type_ffn = nn.Sequential(
            nn.Linear(hidden_dim, ffn_dim), nn.GELU(),
            nn.Dropout(dropout), nn.Linear(ffn_dim, hidden_dim))
        self.layer_norm = nn.LayerNorm(hidden_dim)
        self.use_pretrained = use_pretrained_encoder
        self._cached_output = None

    def _init_pretrained_embeddings(self, encoder_name):
        tokenizer = AutoTokenizer.from_pretrained(encoder_name)
        encoder = AutoModel.from_pretrained(encoder_name)
        encoder.eval()
        embeddings = []
        with torch.no_grad():
            for etype in self.entity_types:
                desc = self.type_descriptions[etype]
                enc = tokenizer(desc, return_tensors="pt", max_length=64,
                                padding="max_length", truncation=True)
                out = encoder(**enc)
                embeddings.append(out.last_hidden_state[:, 0, :].squeeze(0))
        self.register_buffer("pretrained_type_embeddings", torch.stack(embeddings, dim=0))
        del encoder, tokenizer

    def train(self, mode=True):
        if mode and not self.training:
            self._cached_output = None
        return super().train(mode)

    def clear_cache(self):
        self._cached_output = None

    def forward(self, device=None):
        if not self.training and self._cached_output is not None:
            cached = self._cached_output
            if device is not None and cached.device != device:
                cached = cached.to(device)
            return cached
        if self.use_pretrained:
            raw = self.pretrained_type_embeddings
        else:
            indices = torch.arange(self.num_types, device=device or self.type_ffn[0].weight.device)
            raw = self.type_embeddings(indices)
        projected = self.layer_norm(self.type_ffn(raw) + raw)
        if not self.training:
            self._cached_output = projected.detach()
        return projected

    def get_type_index(self, entity_type): return self.type_to_idx.get(entity_type, -1)

log("Loaded: SpanRepresentationLayer, SpanPruner, EntityTypeEncoder")

In [ ]:
# ============================================================================
# [6] SpanEntityMatcherV11_1 — stabilized bi-encoder matching
# [6b] SpanEvidenceAggregator — logsumexp / amax / sum / mean
# [9] GatedSpanTokenFusion — dynamic logit blending
# BCELanguageLoss + create_span_type_labels — KoGNER §2.2
# ============================================================================

class SpanEntityMatcherV11_1(nn.Module):
    """Bi-encoder span-entity matching (v11.1 — LayerNorm stabilized, no sigmoid)."""
    def __init__(self, hidden_dim=768, num_entity_types=3,
                 temperature=1.0, span_logit_temperature=1.5):
        super().__init__()
        self.hidden_dim = hidden_dim
        self.num_entity_types = num_entity_types
        self.temperature = temperature
        self.span_logit_temperature = span_logit_temperature
        self.span_proj = nn.Sequential(nn.Linear(hidden_dim, hidden_dim), nn.LayerNorm(hidden_dim))
        self.type_proj = nn.Sequential(nn.Linear(hidden_dim, hidden_dim), nn.LayerNorm(hidden_dim))
        self.matching_logit_norm = nn.LayerNorm(num_entity_types)

    def forward(self, span_repr, entity_type_repr, span_mask=None):
        s = self.span_proj(span_repr)
        n = self.type_proj(entity_type_repr)
        logits = torch.matmul(s, n.t()) / self.temperature
        if span_mask is not None:
            mask = span_mask.unsqueeze(-1)
            logits = logits * mask + (-1e9) * (1 - mask)
        return self.matching_logit_norm(logits)

    def compute_matching_logits(self, span_repr, entity_type_repr, span_mask=None):
        s = self.span_proj(span_repr)
        n = self.type_proj(entity_type_repr)
        logits = torch.matmul(s, n.t()) / self.temperature
        if span_mask is not None:
            mask = span_mask.unsqueeze(-1)
            logits = logits * mask + (-1e9) * (1 - mask)
        return logits


# ── [6b] Span Evidence Aggregator ────────────────────────────────────────
def _build_scatter_indices(span_indices, num_types, seq_len, num_bio_labels,
                           type_to_bio, device):
    dst_list, src_list = [], []
    for span_idx, (start, end) in enumerate(span_indices):
        for type_idx in range(num_types):
            b_label, i_label = type_to_bio[type_idx]
            src_flat = span_idx * num_types + type_idx
            if start < seq_len:
                dst_list.append(start * num_bio_labels + b_label)
                src_list.append(src_flat)
            for pos in range(start + 1, min(end + 1, seq_len)):
                dst_list.append(pos * num_bio_labels + i_label)
                src_list.append(src_flat)
    return (torch.tensor(dst_list, dtype=torch.long, device=device),
            torch.tensor(src_list, dtype=torch.long, device=device))


class SpanEvidenceAggregator(nn.Module):
    """Aggregates span-level matching logits into token-level BIO logits."""
    def __init__(self, method="logsumexp", num_bio_labels=7, o_bias=0.1):
        super().__init__()
        assert method in ("amax", "sum", "mean", "logsumexp")
        self.method = method
        self.num_bio_labels = num_bio_labels
        self.o_bias = o_bias
        self.type_to_bio = [(1, 2), (3, 4), (5, 6)]

    def forward(self, matching_logits, span_indices, seq_len, span_logit_temperature=1.5):
        batch_size = matching_logits.size(0)
        num_types = min(matching_logits.size(2), len(self.type_to_bio))
        device = matching_logits.device
        flat_dst, flat_src = _build_scatter_indices(
            span_indices, num_types, seq_len, self.num_bio_labels, self.type_to_bio, device)
        if self.method == "logsumexp":
            return self._logsumexp_agg(matching_logits, flat_dst, flat_src,
                                       seq_len, batch_size, device, span_logit_temperature)
        elif self.method == "amax":
            return self._scatter_agg(matching_logits, flat_dst, flat_src,
                                     seq_len, batch_size, device, span_logit_temperature, "amax")
        elif self.method == "sum":
            return self._scatter_agg(matching_logits, flat_dst, flat_src,
                                     seq_len, batch_size, device, span_logit_temperature, "sum")
        else:
            return self._mean_agg(matching_logits, flat_dst, flat_src,
                                  seq_len, batch_size, device, span_logit_temperature)

    def _gather_src(self, matching_logits, flat_src):
        return matching_logits.reshape(matching_logits.size(0), -1)[:, flat_src]

    def _logsumexp_agg(self, matching_logits, flat_dst, flat_src, seq_len,
                       batch_size, device, temperature):
        K = flat_dst.size(0)
        S = seq_len * self.num_bio_labels
        if K == 0:
            out = torch.zeros(batch_size, seq_len, self.num_bio_labels, device=device)
            out[:, :, 0] = self.o_bias
            return out / temperature
        src_vals = self._gather_src(matching_logits, flat_src)
        dst_exp = flat_dst.unsqueeze(0).expand(batch_size, K)
        slot_max = torch.full((batch_size, S), -1e9, device=device)
        slot_max.scatter_reduce_(1, dst_exp, src_vals, reduce="amax", include_self=True)
        gathered_max = slot_max.gather(1, dst_exp)
        exp_vals = torch.exp(src_vals - gathered_max)
        slot_exp_sum = torch.zeros(batch_size, S, device=device)
        slot_exp_sum.scatter_reduce_(1, dst_exp, exp_vals, reduce="sum", include_self=True)
        ones = torch.ones(batch_size, K, device=device)
        slot_count = torch.zeros(batch_size, S, device=device)
        slot_count.scatter_reduce_(1, dst_exp, ones, reduce="sum", include_self=True)
        has_spans = (slot_count > 0)
        token_logits_flat = torch.where(has_spans,
            slot_max + torch.log(slot_exp_sum.clamp(min=1e-8)),
            torch.zeros_like(slot_max))
        token_logits = token_logits_flat.view(batch_size, seq_len, self.num_bio_labels)
        token_logits[:, :, 0] = self.o_bias
        return token_logits / temperature

    def _scatter_agg(self, matching_logits, flat_dst, flat_src, seq_len,
                     batch_size, device, temperature, reduce):
        K = flat_dst.size(0)
        S = seq_len * self.num_bio_labels
        token_logits_flat = torch.zeros(batch_size, S, device=device)
        o_indices = torch.arange(0, S, self.num_bio_labels, device=device)
        token_logits_flat[:, o_indices] = self.o_bias
        if K > 0:
            src_vals = self._gather_src(matching_logits, flat_src)
            dst_exp = flat_dst.unsqueeze(0).expand(batch_size, K)
            token_logits_flat.scatter_reduce_(1, dst_exp, src_vals, reduce=reduce, include_self=True)
        return token_logits_flat.view(batch_size, seq_len, self.num_bio_labels) / temperature

    def _mean_agg(self, matching_logits, flat_dst, flat_src, seq_len,
                  batch_size, device, temperature):
        K = flat_dst.size(0)
        S = seq_len * self.num_bio_labels
        if K == 0:
            out = torch.zeros(batch_size, seq_len, self.num_bio_labels, device=device)
            out[:, :, 0] = self.o_bias
            return out / temperature
        src_vals = self._gather_src(matching_logits, flat_src)
        dst_exp = flat_dst.unsqueeze(0).expand(batch_size, K)
        token_sum = torch.zeros(batch_size, S, device=device)
        token_sum.scatter_reduce_(1, dst_exp, src_vals, reduce="sum", include_self=True)
        ones = torch.ones(batch_size, K, device=device)
        token_count = torch.zeros(batch_size, S, device=device)
        token_count.scatter_reduce_(1, dst_exp, ones, reduce="sum", include_self=True)
        has_spans = (token_count > 0)
        token_logits_flat = torch.where(has_spans,
            token_sum / token_count.clamp(min=1.0), torch.zeros_like(token_sum))
        token_logits = token_logits_flat.view(batch_size, seq_len, self.num_bio_labels)
        token_logits[:, :, 0] = self.o_bias
        return token_logits / temperature


# ── [9] Gated Span-Token Fusion ──────────────────────────────────────────
class GatedSpanTokenFusion(nn.Module):
    """Gated fusion of NER head logits and span-derived token logits."""
    def __init__(self, num_labels=7, gate_hidden_dim=None, dropout=0.1):
        super().__init__()
        if gate_hidden_dim is not None and gate_hidden_dim > 0:
            self.gate = nn.Sequential(
                nn.Linear(num_labels * 2, gate_hidden_dim), nn.GELU(),
                nn.Dropout(dropout), nn.Linear(gate_hidden_dim, num_labels), nn.Sigmoid())
        else:
            self.gate = nn.Sequential(nn.Linear(num_labels * 2, num_labels), nn.Sigmoid())

    def forward(self, ner_logits, span_token_logits):
        gate_input = torch.cat([ner_logits, span_token_logits], dim=-1)
        gate_values = self.gate(gate_input)
        return gate_values * ner_logits + (1 - gate_values) * span_token_logits


# ── BCELanguageLoss [KoGNER §2.2] ────────────────────────────────────────
class BCELanguageLoss(nn.Module):
    def __init__(self, pos_weight=None):
        super().__init__()
        if pos_weight is not None:
            self.criterion = nn.BCEWithLogitsLoss(
                pos_weight=torch.tensor([pos_weight]), reduction="none")
        else:
            self.criterion = nn.BCEWithLogitsLoss(reduction="none")

    def forward(self, matching_logits, span_labels, span_mask=None):
        loss = self.criterion(matching_logits, span_labels.float())
        if span_mask is not None:
            mask = span_mask.unsqueeze(-1).expand_as(loss)
            loss = loss * mask
            return loss.sum() / mask.sum().clamp(min=1.0)
        return loss.mean()


def create_span_type_labels(ner_labels, span_indices, num_types=3):
    """Create binary span-type labels for BCE language loss."""
    batch_size = ner_labels.size(0)
    device = ner_labels.device
    num_spans = len(span_indices)
    labels = torch.zeros(batch_size, num_spans, num_types, device=device)
    b_to_type = {1: 0, 3: 1, 5: 2}
    for b in range(batch_size):
        for s_idx, (start, end) in enumerate(span_indices):
            label_start = ner_labels[b, start].item()
            if label_start <= 0 or label_start == -100:
                continue
            if label_start in b_to_type:
                type_idx = b_to_type[label_start]
                expected_i = label_start + 1
                valid = True
                for pos in range(start + 1, end + 1):
                    if pos >= ner_labels.size(1) or ner_labels[b, pos].item() != expected_i:
                        valid = False; break
                if valid:
                    labels[b, s_idx, type_idx] = 1.0
    return labels

log("Loaded: SpanEntityMatcherV11_1, SpanEvidenceAggregator, GatedSpanTokenFusion, BCELanguageLoss")

In [ ]:
# ============================================================================
# [10] CRFDecoder — Viterbi + BIO constraints
# [7] KGTeacher — GraphTransformer + TransR → H = [Z, Z', Z'']
# ============================================================================

class CRFDecoder(nn.Module):
    """Linear-chain CRF for BIO-tagged NER with hard transition constraints."""
    def __init__(self, num_labels=7, use_hard_constraints=True):
        super().__init__()
        self.num_labels = num_labels
        self.use_hard_constraints = use_hard_constraints
        self.transitions = nn.Parameter(torch.randn(num_labels, num_labels))
        self.start_transitions = nn.Parameter(torch.randn(num_labels))
        self.end_transitions = nn.Parameter(torch.randn(num_labels))
        self._init_transitions()

    def _init_transitions(self):
        nn.init.uniform_(self.transitions, -0.1, 0.1)
        nn.init.uniform_(self.start_transitions, -0.1, 0.1)
        nn.init.uniform_(self.end_transitions, -0.1, 0.1)
        if self.use_hard_constraints:
            self._apply_bio_constraints()

    def _apply_bio_constraints(self):
        IMPOSSIBLE = -10000.0
        i_labels = [2, 4, 6]; b_labels = [1, 3, 5]
        i_to_b = {2: 1, 4: 3, 6: 5}
        with torch.no_grad():
            for i_label in i_labels:
                corresponding_b = i_to_b[i_label]
                self.transitions.data[i_label, 0] = IMPOSSIBLE
                for other_b in b_labels:
                    if other_b != corresponding_b:
                        self.transitions.data[i_label, other_b] = IMPOSSIBLE
                for other_i in i_labels:
                    if other_i != i_label:
                        self.transitions.data[i_label, other_i] = IMPOSSIBLE
                self.start_transitions.data[i_label] = IMPOSSIBLE

    def forward(self, emissions, labels, mask=None):
        if mask is None:
            mask = torch.ones_like(labels, dtype=torch.bool)
        else:
            mask = mask.bool()
        safe_labels = labels.clone(); safe_labels[safe_labels == -100] = 0
        label_mask = (labels != -100); mask = mask & label_mask
        gold_score = self._compute_score(emissions, safe_labels, mask)
        forward_score = self._compute_log_partition(emissions, mask)
        return (forward_score - gold_score).mean()

    def decode(self, emissions, mask=None):
        if mask is None:
            mask = torch.ones(emissions.shape[:2], dtype=torch.bool, device=emissions.device)
        else:
            mask = mask.bool()
        batch_size, seq_len, _ = emissions.shape
        best_paths = []
        for b in range(batch_size):
            seq_mask = mask[b]; seq_len_b = seq_mask.sum().item()
            if seq_len_b == 0:
                best_paths.append([0] * seq_len); continue
            path = self._viterbi_decode(emissions[b, :seq_len_b])
            best_paths.append(path + [0] * (seq_len - seq_len_b))
        return best_paths

    def _compute_score(self, emissions, labels, mask):
        batch_size, seq_len, _ = emissions.shape
        score = self.start_transitions[labels[:, 0]]
        score = score + emissions[:, 0].gather(1, labels[:, 0].unsqueeze(1)).squeeze(1)
        for t in range(1, seq_len):
            trans = self.transitions[labels[:, t], labels[:, t - 1]]
            emit = emissions[:, t].gather(1, labels[:, t].unsqueeze(1)).squeeze(1)
            score = score + (trans + emit) * mask[:, t].float()
        seq_lengths = mask.sum(dim=1).long()
        last_labels = labels.gather(1, (seq_lengths - 1).unsqueeze(1)).squeeze(1)
        return score + self.end_transitions[last_labels]

    def _compute_log_partition(self, emissions, mask):
        batch_size, seq_len, num_labels = emissions.shape
        alpha = self.start_transitions + emissions[:, 0]
        for t in range(1, seq_len):
            emit = emissions[:, t].unsqueeze(1)
            trans = self.transitions.unsqueeze(0)
            scores = alpha.unsqueeze(2) + trans + emit
            new_alpha = torch.logsumexp(scores, dim=1)
            m = mask[:, t].unsqueeze(1).float()
            alpha = new_alpha * m + alpha * (1 - m)
        return torch.logsumexp(alpha + self.end_transitions, dim=1)

    def _viterbi_decode(self, emissions):
        seq_len, num_labels = emissions.shape
        viterbi = self.start_transitions + emissions[0]
        backpointers = []
        for t in range(1, seq_len):
            scores = viterbi.unsqueeze(1) + self.transitions
            best_scores, best_ids = scores.max(dim=0)
            viterbi = best_scores + emissions[t]
            backpointers.append(best_ids)
        viterbi = viterbi + self.end_transitions
        best_last = viterbi.argmax().item()
        best_path = [best_last]
        for bp in reversed(backpointers):
            best_path.append(bp[best_path[-1]].item())
        best_path.reverse()
        return best_path


# ── [7] KG Teacher ───────────────────────────────────────────────────────
class GraphTransformerLayer(nn.Module):
    def __init__(self, hidden_dim, num_heads=4, dropout=0.1):
        super().__init__()
        self.hidden_dim = hidden_dim
        self.num_heads = num_heads
        self.head_dim = hidden_dim // num_heads
        assert hidden_dim % num_heads == 0
        self.q_proj = nn.Linear(hidden_dim, hidden_dim)
        self.k_proj = nn.Linear(hidden_dim, hidden_dim)
        self.v_proj = nn.Linear(hidden_dim, hidden_dim)
        self.out_proj = nn.Linear(hidden_dim, hidden_dim)
        self.ffn = nn.Sequential(nn.Linear(hidden_dim, hidden_dim * 4), nn.GELU(),
                                 nn.Dropout(dropout), nn.Linear(hidden_dim * 4, hidden_dim))
        self.norm1 = nn.LayerNorm(hidden_dim)
        self.norm2 = nn.LayerNorm(hidden_dim)
        self.dropout = nn.Dropout(dropout)

    def forward(self, node_features, adjacency):
        N = node_features.size(0); residual = node_features
        Q = self.q_proj(node_features).view(N, self.num_heads, self.head_dim)
        K = self.k_proj(node_features).view(N, self.num_heads, self.head_dim)
        V = self.v_proj(node_features).view(N, self.num_heads, self.head_dim)
        scale = math.sqrt(self.head_dim)
        attn = torch.einsum("nhd,mhd->hnm", Q, K) / scale
        adj_mask = adjacency.unsqueeze(0).expand(self.num_heads, -1, -1)
        attn = attn.masked_fill(adj_mask == 0, float("-inf"))
        eye = torch.eye(N, device=node_features.device).unsqueeze(0)
        attn = attn.masked_fill((adj_mask == 0) & (eye == 0), float("-inf"))
        attn = torch.nan_to_num(self.dropout(F.softmax(attn, dim=-1)), nan=0.0)
        out = self.out_proj(torch.einsum("hnm,mhd->nhd", attn, V).reshape(N, self.hidden_dim))
        out = self.norm1(self.dropout(out) + residual)
        return self.norm2(self.ffn(out) + out)


class GraphTransformer(nn.Module):
    def __init__(self, input_dim=768, hidden_dim=256, num_layers=2,
                 num_heads=4, num_classes=4, dropout=0.1):
        super().__init__()
        self.input_proj = nn.Linear(input_dim, hidden_dim)
        self.layers = nn.ModuleList([GraphTransformerLayer(hidden_dim, num_heads, dropout)
                                     for _ in range(num_layers)])
        self.classifier = nn.Linear(hidden_dim, num_classes)
        self.hidden_dim = hidden_dim

    def forward(self, node_features, adjacency):
        h = self.input_proj(node_features)
        for layer in self.layers: h = layer(h, adjacency)
        return h, self.classifier(h)

    def get_spatial_embeddings(self, node_features, adjacency):
        h = self.input_proj(node_features)
        for layer in self.layers: h = layer(h, adjacency)
        return h


class TransR(nn.Module):
    def __init__(self, num_entities, num_relations, entity_dim=256,
                 relation_dim=256, margin=1.0):
        super().__init__()
        self.entity_dim = entity_dim; self.relation_dim = relation_dim
        self.entity_embeddings = nn.Embedding(num_entities, entity_dim)
        self.relation_embeddings = nn.Embedding(num_relations, relation_dim)
        self.projection_matrices = nn.Embedding(num_relations, entity_dim * relation_dim)
        self.margin = margin
        nn.init.xavier_uniform_(self.entity_embeddings.weight)
        nn.init.xavier_uniform_(self.relation_embeddings.weight)
        nn.init.xavier_uniform_(self.projection_matrices.weight)

    def _project(self, entity_emb, relation_idx):
        M_r = self.projection_matrices(relation_idx).view(-1, self.entity_dim, self.relation_dim)
        return F.normalize(torch.bmm(entity_emb.unsqueeze(1), M_r).squeeze(1), dim=-1)

    def forward(self, head_idx, relation_idx, tail_idx):
        h = self.entity_embeddings(head_idx); t = self.entity_embeddings(tail_idx)
        r = self.relation_embeddings(relation_idx)
        return torch.norm(self._project(h, relation_idx) + r - self._project(t, relation_idx), p=2, dim=-1)

    def get_entity_embeddings(self): return self.entity_embeddings.weight


class KGTeacher(nn.Module):
    """KG Teacher: H = [Z, Z', Z''] for knowledge distillation."""
    def __init__(self, num_entities, num_relations=4, text_dim=768,
                 spatial_dim=256, logical_dim=256, gnn_layers=2,
                 gnn_heads=4, num_entity_classes=4, dropout=0.1):
        super().__init__()
        self.text_dim = text_dim; self.spatial_dim = spatial_dim
        self.logical_dim = logical_dim
        self.combined_dim = text_dim + spatial_dim + logical_dim
        self.gnn = GraphTransformer(text_dim, spatial_dim, gnn_layers,
                                    gnn_heads, num_entity_classes, dropout)
        self.transr = TransR(num_entities, num_relations, logical_dim, logical_dim)
        self.num_entities = num_entities

    def forward(self, text_embeddings, adjacency, entity_indices=None):
        Z = text_embeddings
        Z_prime = self.gnn.get_spatial_embeddings(Z, adjacency)
        Z_double_prime = self.transr.get_entity_embeddings()
        num_entities = Z.size(0)
        if Z_double_prime.size(0) > num_entities:
            Z_double_prime = Z_double_prime[:num_entities]
        elif Z_double_prime.size(0) < num_entities:
            padding = torch.zeros(num_entities - Z_double_prime.size(0),
                                  Z_double_prime.size(1), device=Z_double_prime.device)
            Z_double_prime = torch.cat([Z_double_prime, padding], dim=0)
        H = torch.cat([Z, Z_prime, Z_double_prime], dim=-1)
        if entity_indices is not None: H = H[entity_indices]
        return H

    def get_combined_dim(self): return self.combined_dim

    def pretrain_gnn(self, text_embeddings, adjacency, node_labels, epochs=50, lr=1e-3):
        optimizer = torch.optim.Adam(self.gnn.parameters(), lr=lr)
        criterion = nn.CrossEntropyLoss()
        self.gnn.train(); losses = []
        for epoch in range(epochs):
            optimizer.zero_grad()
            _, logits = self.gnn(text_embeddings, adjacency)
            loss = criterion(logits, node_labels); loss.backward(); optimizer.step()
            losses.append(loss.item())
            if (epoch + 1) % 10 == 0:
                acc = (logits.argmax(dim=-1) == node_labels).float().mean().item()
                print(f"  GNN Pretrain Epoch {epoch+1}/{epochs}: loss={loss.item():.4f}, acc={acc:.4f}")
        for param in self.gnn.parameters(): param.requires_grad = False
        return {"final_loss": losses[-1], "num_epochs": epochs}

    def pretrain_transr(self, triples, epochs=100, lr=1e-3, negative_samples=5):
        optimizer = torch.optim.Adam(self.transr.parameters(), lr=lr)
        device = self.transr.entity_embeddings.weight.device
        self.transr.train(); losses = []
        for epoch in range(epochs):
            optimizer.zero_grad()
            total_loss = torch.tensor(0.0, device=device)
            heads = torch.tensor([t[0] for t in triples], device=device)
            rels = torch.tensor([t[1] for t in triples], device=device)
            tails = torch.tensor([t[2] for t in triples], device=device)
            pos_scores = self.transr(heads, rels, tails)
            for _ in range(negative_samples):
                neg_tails = torch.randint(0, self.num_entities, (len(triples),), device=device)
                neg_scores = self.transr(heads, rels, neg_tails)
                total_loss = total_loss + F.relu(self.transr.margin + pos_scores - neg_scores).mean()
            total_loss = total_loss / negative_samples
            total_loss.backward(); optimizer.step(); losses.append(total_loss.item())
            if (epoch + 1) % 20 == 0:
                print(f"  TransR Pretrain Epoch {epoch+1}/{epochs}: loss={total_loss.item():.4f}")
        for param in self.transr.parameters(): param.requires_grad = False
        return {"final_loss": losses[-1], "num_epochs": epochs}

log("Loaded: CRFDecoder, GraphTransformer, TransR, KGTeacher")

In [ ]:
# ============================================================================
# LOSSES: FocalLoss, BoundaryLoss, EntityTypeLoss, KnowledgeAlignmentLoss,
#          ContrastiveEntityLoss, KGDistillationLoss, CompositeLossV12
# TRAINING SCHEDULE: 3-stage (warmup → structural → contrastive)
# ============================================================================

class FocalLoss(nn.Module):
    """Focal Loss: FL(p_t) = -alpha_t * (1 - p_t)^gamma * log(p_t)."""
    def __init__(self, gamma=2.0, alpha=None, ignore_index=-100, reduction="mean"):
        super().__init__()
        self.gamma = gamma; self.alpha = alpha
        self.ignore_index = ignore_index; self.reduction = reduction

    def forward(self, logits, targets):
        if logits.dim() == 3:
            B, L, C = logits.shape; logits = logits.reshape(-1, C); targets = targets.reshape(-1)
        mask = targets != self.ignore_index; logits = logits[mask]; targets = targets[mask]
        if logits.numel() == 0:
            return torch.tensor(0.0, device=logits.device, requires_grad=True)
        log_probs = F.log_softmax(logits, dim=-1); probs = torch.exp(log_probs)
        targets_one_hot = F.one_hot(targets, num_classes=logits.size(-1)).float()
        p_t = (probs * targets_one_hot).sum(dim=-1)
        log_p_t = (log_probs * targets_one_hot).sum(dim=-1)
        focal_weight = (1.0 - p_t) ** self.gamma
        if self.alpha is not None:
            alpha_t = self.alpha.to(logits.device)[targets]
            focal_weight = alpha_t * focal_weight
        loss = -focal_weight * log_p_t
        return loss.mean() if self.reduction == "mean" else (loss.sum() if self.reduction == "sum" else loss)


class BoundaryLoss(nn.Module):
    def __init__(self, num_classes=3, ignore_index=-100):
        super().__init__()
        self.criterion = nn.CrossEntropyLoss(ignore_index=ignore_index)
    def forward(self, logits, targets):
        if logits.dim() == 3: logits = logits.reshape(-1, logits.size(-1)); targets = targets.reshape(-1)
        return self.criterion(logits, targets)


class EntityTypeLoss(nn.Module):
    def __init__(self, num_classes=4, ignore_index=-100):
        super().__init__()
        self.criterion = nn.CrossEntropyLoss(ignore_index=ignore_index)
    def forward(self, logits, targets):
        if logits.dim() == 3: logits = logits.reshape(-1, logits.size(-1)); targets = targets.reshape(-1)
        return self.criterion(logits, targets)


class KnowledgeAlignmentLoss(nn.Module):
    def __init__(self, margin=0.2):
        super().__init__()
        self.margin = margin
    def forward(self, token_repr, knowledge_repr, labels, attention_mask=None):
        cos_sim = F.cosine_similarity(token_repr, knowledge_repr, dim=-1)
        entity_mask = (labels > 0).float(); o_mask = (labels == 0).float()
        if attention_mask is not None:
            entity_mask = entity_mask * attention_mask.float()
            o_mask = o_mask * attention_mask.float()
        entity_loss = -cos_sim * entity_mask
        o_loss = F.relu(cos_sim - self.margin) * o_mask
        num_tokens = (entity_mask + o_mask).sum().clamp(min=1.0)
        return (entity_loss.sum() + o_loss.sum()) / num_tokens


class ContrastiveEntityLoss(nn.Module):
    """Supervised contrastive loss for entity embeddings."""
    def __init__(self, temperature=0.07, margin=1.0):
        super().__init__()
        self.temperature = temperature; self.margin = margin

    def forward(self, token_repr, ner_labels, attention_mask=None):
        batch_size, seq_len, hidden_dim = token_repr.shape
        device = token_repr.device
        type_map = torch.zeros(10, dtype=torch.long, device=device)
        type_map[1] = 1; type_map[2] = 1; type_map[3] = 2
        type_map[4] = 2; type_map[5] = 3; type_map[6] = 3
        flat_repr = token_repr.reshape(-1, hidden_dim)
        flat_labels = ner_labels.reshape(-1)
        valid = (flat_labels > 0) & (flat_labels != -100)
        if attention_mask is not None:
            valid = valid & attention_mask.reshape(-1).bool()
        valid_indices = valid.nonzero(as_tuple=True)[0]
        if valid_indices.numel() < 2:
            return torch.tensor(0.0, device=device, requires_grad=True)
        valid_repr = flat_repr[valid_indices]
        valid_types = type_map[flat_labels[valid_indices].clamp(0, 9)]
        N = valid_repr.size(0)
        if N < 2: return torch.tensor(0.0, device=device, requires_grad=True)
        max_samples = 512
        if N > max_samples:
            perm = torch.randperm(N, device=device)[:max_samples]
            valid_repr = valid_repr[perm]; valid_types = valid_types[perm]; N = max_samples
        valid_repr = F.normalize(valid_repr, dim=1)
        sim = torch.mm(valid_repr, valid_repr.t()) / self.temperature
        type_match = (valid_types.unsqueeze(0) == valid_types.unsqueeze(1))
        self_mask = ~torch.eye(N, dtype=torch.bool, device=device)
        pos_mask = type_match & self_mask
        if pos_mask.sum() == 0:
            return torch.tensor(0.0, device=device, requires_grad=True)
        neg_mask = ~type_match & self_mask
        exp_sim = torch.exp(sim)
        pos_sim = (exp_sim * pos_mask.float()).sum(dim=1)
        neg_sim = (exp_sim * neg_mask.float()).sum(dim=1)
        loss = -torch.log((pos_sim / (pos_sim + neg_sim + 1e-8)).clamp(min=1e-8))
        return loss.mean()


class BCELanguageLoss(nn.Module):
    def __init__(self, pos_weight=5.0):
        super().__init__()
        self.pos_weight = pos_weight
    def forward(self, logits, targets, mask=None):
        probs = torch.sigmoid(logits)
        pos_w = torch.tensor(self.pos_weight, device=logits.device)
        bce = -(pos_w * targets * torch.log(probs + 1e-8) + (1 - targets) * torch.log(1 - probs + 1e-8))
        if mask is not None:
            bce = bce * mask.unsqueeze(-1).float()
            return bce.sum() / mask.sum().clamp(min=1.0)
        return bce.mean()


class KGDistillationLoss(nn.Module):
    def __init__(self, hidden_dim=768, distill_dim=256, kg_teacher_dim=None, temperature=2.0):
        super().__init__()
        self.temperature = temperature
        student_in = hidden_dim
        teacher_in = kg_teacher_dim if kg_teacher_dim else hidden_dim
        self.student_proj = nn.Sequential(nn.Linear(student_in, distill_dim), nn.GELU())
        self.teacher_proj = nn.Sequential(nn.Linear(teacher_in, distill_dim), nn.GELU())
    def forward(self, student_repr, teacher_repr, mask=None):
        s = self.student_proj(student_repr)
        t = self.teacher_proj(teacher_repr)
        mse = F.mse_loss(s, t, reduction="none").mean(dim=-1)
        if mask is not None:
            mse = mse * mask.float()
            return mse.sum() / mask.sum().clamp(min=1.0)
        return mse.mean()


class CompositeLossV12(nn.Module):
    """v12 composite loss: KoGNER core (BCE, KG distill) + extensions (focal, boundary, type, align, contrastive)."""
    def __init__(self, config):
        super().__init__()
        self.lambda_language = config.get("lambda_language", 0.5)
        self.lambda_distillation = config.get("lambda_distillation", 0.15)
        self.lambda_ner = config.get("lambda_ner", 2.0)  # TASK_5: increased from 1.0
        self.lambda_boundary = config.get("lambda_boundary", 0.3)
        self.lambda_type = config.get("lambda_type", 0.3)
        self.lambda_align = config.get("lambda_align", 0.1)
        self.lambda_contrastive = config.get("lambda_contrastive", 0.2)
        # FIX A: Always initialize all loss modules (stage transitions activate them dynamically)
        self.use_language_loss = config.get("use_language_loss", True)
        self.language_loss = BCELanguageLoss(pos_weight=config.get("bce_pos_weight", 5.0))
        self.use_distillation = config.get("use_distillation_loss", False)
        self.distillation_loss = KGDistillationLoss(
            hidden_dim=config.get("hidden_dim", 768),
            distill_dim=config.get("distill_dim", 256),
            kg_teacher_dim=config.get("kg_teacher_dim", None),
            temperature=config.get("distillation_temperature", 2.0))
        use_focal = config.get("use_focal_loss", True)
        gamma = config.get("focal_gamma", 2.0)
        alpha_weights = config.get("focal_alpha", None)
        if use_focal:
            alpha_tensor = torch.tensor(alpha_weights, dtype=torch.float32) if alpha_weights else None
            self.ner_loss = FocalLoss(gamma=gamma, alpha=alpha_tensor)
        else:
            self.ner_loss = nn.CrossEntropyLoss(ignore_index=-100)
        self.boundary_loss = BoundaryLoss()
        self.type_loss = EntityTypeLoss()
        self.align_loss = KnowledgeAlignmentLoss(margin=config.get("alignment_margin", 0.2))
        self.use_contrastive = config.get("use_contrastive_loss", False)
        self.contrastive_loss = ContrastiveEntityLoss(
            temperature=config.get("contrastive_temperature", 0.07),
            margin=config.get("contrastive_margin", 1.0))

    def forward(self, ner_logits, ner_labels,
                boundary_logits=None, boundary_labels=None,
                type_logits=None, type_labels=None,
                token_repr=None, knowledge_repr=None, attention_mask=None,
                crf_loss=None,
                matching_logits=None, span_type_labels=None,
                span_mask=None, span_repr=None, kg_teacher_embeddings=None):
        losses = {}
        total = torch.tensor(0.0, device=ner_logits.device, requires_grad=True)
        if self.use_language_loss and matching_logits is not None and span_type_labels is not None:
            loss_lang = self.language_loss(matching_logits, span_type_labels, span_mask)
            losses["language"] = loss_lang; total = total + self.lambda_language * loss_lang
        if self.use_distillation and span_repr is not None:
            if kg_teacher_embeddings is not None:
                loss_dist = self.distillation_loss(span_repr, kg_teacher_embeddings, span_mask)
            elif knowledge_repr is not None:
                loss_dist = self.distillation_loss(token_repr, knowledge_repr, attention_mask)
            else:
                loss_dist = torch.tensor(0.0, device=ner_logits.device)
            losses["distillation"] = loss_dist; total = total + self.lambda_distillation * loss_dist
        # TASK_7: Normalize CRF loss by batch_size instead of seq_len
        if crf_loss is not None:
            batch_size = ner_logits.size(0)
            crf_loss_normalized = (crf_loss / batch_size).clamp(min=0.0)
            losses["crf"] = crf_loss_normalized; total = total + self.lambda_ner * crf_loss_normalized
        else:
            loss_ner = self.ner_loss(ner_logits, ner_labels)
            losses["ner"] = loss_ner; total = total + self.lambda_ner * loss_ner
        if boundary_logits is not None and boundary_labels is not None:
            lb = self.boundary_loss(boundary_logits, boundary_labels)
            losses["boundary"] = lb; total = total + self.lambda_boundary * lb
        if type_logits is not None and type_labels is not None:
            lt = self.type_loss(type_logits, type_labels)
            losses["type"] = lt; total = total + self.lambda_type * lt
        # FIX 3: Shape guard — only compute alignment when shapes match
        if token_repr is not None and knowledge_repr is not None:
            if token_repr.shape == knowledge_repr.shape:
                la = self.align_loss(token_repr, knowledge_repr, ner_labels, attention_mask)
                losses["alignment"] = la; total = total + self.lambda_align * la
        if self.use_contrastive and token_repr is not None:
            lc = self.contrastive_loss(token_repr, ner_labels, attention_mask)
            losses["contrastive"] = lc; total = total + self.lambda_contrastive * lc
        losses["total"] = total
        return losses


# ── Training Schedule ─────────────────────────────────────────────────────
@dataclass
class TrainingStage:
    name: str
    epochs: int
    learning_rate_multiplier: float = 1.0
    active_losses: List[str] = field(default_factory=list)
    description: str = ""


class TrainingSchedule:
    """3-stage training schedule: warmup → structural → contrastive."""
    def __init__(self, config=None):
        config = config or {}
        schedule_cfg = config.get("training_schedule", {})
        self.stages = self._build_stages(schedule_cfg)
        self._current_stage_idx = 0; self._epoch_in_stage = 0

    def _build_stages(self, cfg):
        if "stages" in cfg:
            return [TrainingStage(name=s["name"], epochs=s["epochs"],
                learning_rate_multiplier=s.get("lr_multiplier", 1.0),
                active_losses=s.get("active_losses", []),
                description=s.get("description", "")) for s in cfg["stages"]]
        return [
            # TASK_8: Extended warmup from 3 to 5 epochs
            TrainingStage(name="stage1_warmup", epochs=cfg.get("stage1_epochs", 5),
                learning_rate_multiplier=1.0,
                active_losses=["ner", "crf", "language"],
                description="NER + CRF + language (BCE) — stabilize encoder & transitions"),
            # TASK_9: Set LR multiplier to 1.0 (was 0.8)
            TrainingStage(name="stage2a_structural", epochs=cfg.get("stage2a_epochs", 5),
                learning_rate_multiplier=1.0,
                active_losses=["ner", "crf", "boundary", "type", "distillation", "language", "alignment"],
                description="NER + boundary + type + distillation — structured learning"),
            # TASK_9: Set LR multiplier to 1.0 (was 0.6)
            # TASK_6: Removed contrastive from stage2b
            TrainingStage(name="stage2b_contrastive", epochs=cfg.get("stage2b_epochs", 7),
                learning_rate_multiplier=1.0,
                active_losses=["ner", "crf", "language", "boundary", "type", "alignment"],
                description="NER + CRF + language + boundary + type + alignment — refinement"),
        ]

    @property
    def total_epochs(self): return sum(s.epochs for s in self.stages)
    @property
    def num_stages(self): return len(self.stages)
    @property
    def current_stage(self): return self.stages[self._current_stage_idx]

    def get_stage_for_epoch(self, global_epoch):
        cumulative = 0
        for stage in self.stages:
            cumulative += stage.epochs
            if global_epoch <= cumulative: return stage
        return self.stages[-1]

    def is_loss_active(self, loss_name, global_epoch):
        return loss_name in self.get_stage_for_epoch(global_epoch).active_losses

    def get_lr_multiplier(self, global_epoch):
        return self.get_stage_for_epoch(global_epoch).learning_rate_multiplier

    def advance_epoch(self):
        self._epoch_in_stage += 1
        if self._epoch_in_stage >= self.current_stage.epochs:
            self._current_stage_idx = min(self._current_stage_idx + 1, len(self.stages) - 1)
            self._epoch_in_stage = 0

    def summary(self):
        lines = ["Training Schedule:"]
        for i, stage in enumerate(self.stages, 1):
            lines.append(f"  Stage {i}: {stage.name} ({stage.epochs} epochs, LR×{stage.learning_rate_multiplier})")
            lines.append(f"    Active losses: {', '.join(stage.active_losses)}")
            if stage.description:
                lines.append(f"    {stage.description}")
        lines.append(f"Total epochs: {self.total_epochs}")
        return "\n".join(lines)

log("Loaded: CompositeLossV12, TrainingSchedule")
log("TASK_5: lambda_ner increased from 1.0 to 2.0")
log("TASK_7: CRF normalization changed from /seq_len to /batch_size")
log("TASK_8: Stage 1 warmup extended from 3 to 5 epochs")
log("TASK_9: All LR multipliers set to 1.0")
log("TASK_6: Contrastive loss removed from stage2b")

In [ ]:
# ============================================================================
# MAIN MODEL: KoGNERv12 (KnowledgeAwareNERModelV11_6) — self-contained
# ============================================================================

class KnowledgeAwareNERModelV11_6(nn.Module):
    """
    Adapted KoGNER v11.6 Knowledge-Aware Biomedical NER Model.
    Architecture: 1→BioBERT, 2→KnowledgeEnc, 3→Fusion, 4→SpanRepr,
    4b→Pruning, 5→EntityTypeEnc, 6→Matcher, 6b→EvidenceAgg,
    7→KGTeacher(ext), 8→Heads, 9→GatedFusion, 10→CRF.
    """

    def __init__(self, config: Dict):
        super().__init__()
        self.config = config
        model_cfg = config.get("model", {})
        span_cfg = config.get("span_representation", {})
        fusion_cfg = config.get("knowledge_fusion", {})
        crf_cfg = config.get("crf", {})
        features = config.get("features", {})
        biencoder_cfg = config.get("biencoder", {})

        self.hidden_dim = model_cfg.get("hidden_dim", 768)
        self.num_labels = model_cfg.get("num_labels", 7)
        self.num_entity_types = model_cfg.get("num_entity_types", 4)
        self.num_boundary_classes = model_cfg.get("num_boundary_classes", 3)
        dropout = model_cfg.get("dropout_rate", 0.1)

        # Feature flags
        self.use_span = features.get("use_span_representation", True)
        self.use_fusion = features.get("use_knowledge_fusion", True)
        self.use_crf = features.get("use_crf_decoder", True)
        self.use_biencoder = features.get("use_biencoder", True)
        self.use_gated_fusion = features.get("use_gated_span_fusion", True)
        self.use_span_pruning = features.get("use_span_pruning", True)

        # [1] BioBERT Text Encoder
        self.encoder = KnowledgeAwareEncoder(
            encoder_name=model_cfg.get("encoder_name", "dmis-lab/biobert-base-cased-v1.2"),
            hidden_dim=self.hidden_dim, dropout=dropout,
            freeze_encoder=model_cfg.get("freeze_encoder", False))

        # [2] Knowledge Encoder
        ks_cfg = config.get("knowledge_store", {})
        retrieval_cfg = ks_cfg.get("retrieval", {})
        ke_cfg = config.get("knowledge_encoder", {})
        self.knowledge_encoder = KnowledgeEncoder(
            embedding_dim=ks_cfg.get("embedding_dim", 768),
            hidden_dim=self.hidden_dim, dropout=dropout,
            top_k=retrieval_cfg.get("top_k", 5),
            mode=ke_cfg.get("mode", "broadcast"))

        # [3] Knowledge Fusion Layer
        if self.use_fusion:
            fusion_cfg_with_dim = {**fusion_cfg, "hidden_dim": self.hidden_dim}
            self.fusion_layer = KnowledgeFusionLayer(fusion_cfg_with_dim)
        else:
            self.fusion_layer = None

        # [4] Span Representation
        if self.use_span:
            self.span_layer = SpanRepresentationLayer(
                hidden_dim=self.hidden_dim,
                span_ffn_dim=span_cfg.get("span_ffn_dim", 1024),
                max_span_width=span_cfg.get("max_span_width", 8),
                use_width_embedding=span_cfg.get("use_width_embedding", True),
                width_embedding_dim=150,
                dropout=span_cfg.get("dropout", 0.1),
                pooling=span_cfg.get("pooling", "endpoint"))

        # [4b] Span Pruning
        if self.use_span and self.use_span_pruning:
            self.span_pruner = SpanPruner(
                hidden_dim=self.hidden_dim,
                max_spans_per_sequence=model_cfg.get("max_spans_per_sequence", 100),
                dropout=dropout)

        # [5] Entity Type Encoder
        if self.use_biencoder:
            entity_descs = biencoder_cfg.get(
                "entity_type_descriptions", DEFAULT_ENTITY_TYPE_DESCRIPTIONS)
            self.entity_type_encoder = EntityTypeEncoder(
                entity_type_descriptions=entity_descs,
                hidden_dim=self.hidden_dim,
                encoder_name=model_cfg.get("encoder_name", "dmis-lab/biobert-base-cased-v1.2"),
                use_pretrained_encoder=biencoder_cfg.get("use_pretrained_encoder", True),
                freeze_type_encoder=biencoder_cfg.get("freeze_type_encoder", True),
                ffn_dim=biencoder_cfg.get("ffn_dim", 1024), dropout=dropout)

            # [6] Span-Entity Matcher (v11.1 stabilized)
            self.span_entity_matcher = SpanEntityMatcherV11_1(
                hidden_dim=self.hidden_dim,
                num_entity_types=len(entity_descs),
                temperature=biencoder_cfg.get("temperature", 1.0),
                span_logit_temperature=model_cfg.get("span_logit_temperature", 1.5))

        # [6b] Evidence Aggregation (v11.4)
        if self.use_biencoder and self.use_span:
            self.evidence_aggregator = SpanEvidenceAggregator(
                method=model_cfg.get("span_aggregation_method", "logsumexp"),
                num_bio_labels=self.num_labels)

        # [8] Multi-task Heads
        self.ner_head = nn.Sequential(nn.Dropout(dropout), nn.Linear(self.hidden_dim, self.num_labels))
        self.boundary_head = nn.Sequential(nn.Dropout(dropout), nn.Linear(self.hidden_dim, self.num_boundary_classes))
        self.type_head = nn.Sequential(nn.Dropout(dropout), nn.Linear(self.hidden_dim, self.num_entity_types))

        # [9] Gated Span-Token Fusion (v11.2)
        if self.use_biencoder and self.use_span and self.use_gated_fusion:
            self.span_token_gate = GatedSpanTokenFusion(
                num_labels=self.num_labels,
                gate_hidden_dim=model_cfg.get("gate_hidden_dim", None), dropout=dropout)

        # [10] CRF Decoder
        if self.use_crf:
            self.crf = CRFDecoder(num_labels=self.num_labels,
                use_hard_constraints=crf_cfg.get("use_hard_constraints", True))

        self._log_architecture()

    def _log_architecture(self):
        total_params = sum(p.numel() for p in self.parameters())
        trainable_params = sum(p.numel() for p in self.parameters() if p.requires_grad)
        encoder_params = sum(p.numel() for p in self.encoder.parameters())
        knowledge_params = sum(p.numel() for p in self.knowledge_encoder.parameters())
        span_params = sum(p.numel() for p in self.span_layer.parameters()) if self.use_span else 0
        fusion_params = sum(p.numel() for p in self.fusion_layer.parameters()) if self.use_fusion else 0
        crf_params = sum(p.numel() for p in self.crf.parameters()) if self.use_crf else 0
        biencoder_params = 0
        if self.use_biencoder:
            biencoder_params = (sum(p.numel() for p in self.entity_type_encoder.parameters())
                + sum(p.numel() for p in self.span_entity_matcher.parameters()))
        pruner_params = sum(p.numel() for p in self.span_pruner.parameters()) if self.use_span and self.use_span_pruning else 0
        gate_params = sum(p.numel() for p in self.span_token_gate.parameters()) if (self.use_biencoder and self.use_span and self.use_gated_fusion) else 0
        head_params = (sum(p.numel() for p in self.ner_head.parameters())
            + sum(p.numel() for p in self.boundary_head.parameters())
            + sum(p.numel() for p in self.type_head.parameters()))
        print(f"KoGNERv12 initialized")
        print(f"   [1]  BioBERT Encoder: {self.config.get('model', {}).get('encoder_name', '?')} ({encoder_params:,})")
        print(f"   [2]  Knowledge Encoder: ({knowledge_params:,})")
        print(f"   [3]  Knowledge Fusion: {self.config.get('knowledge_fusion', {}).get('method', 'none')} ({fusion_params:,})")
        print(f"   [4]  Span Repr: {'ON' if self.use_span else 'OFF'} ({span_params:,})")
        print(f"   [4b] Span Pruning: {'ON' if self.use_span_pruning else 'OFF'} ({pruner_params:,})")
        print(f"   [5+6] Bi-Encoder + Matcher: {'ON' if self.use_biencoder else 'OFF'} ({biencoder_params:,})")
        print(f"   [8]  Multi-task Heads: ({head_params:,})")
        print(f"   [9]  Gated Span-Token Fusion: {'ON' if self.use_gated_fusion else 'OFF'} ({gate_params:,})")
        print(f"   [10] CRF Decoder: {'ON' if self.use_crf else 'OFF'} ({crf_params:,})")
        print(f"   Total: {total_params:,} (trainable: {trainable_params:,})")

    def forward(self, input_ids, attention_mask, knowledge_embeddings,
                labels=None, boundary_labels=None, type_labels=None,
                token_type_ids=None, kg_teacher_embeddings=None):
        batch_size, seq_len = input_ids.shape
        outputs = {}

        # [1] BioBERT
        token_repr = self.encoder(input_ids=input_ids, attention_mask=attention_mask,
                                  token_type_ids=token_type_ids)
        outputs["token_repr_raw"] = token_repr

        # [2] Knowledge Encoder
        knowledge_repr = self.knowledge_encoder(knowledge_embeddings=knowledge_embeddings,
                                                seq_len=seq_len, attention_mask=attention_mask)
        outputs["knowledge_repr"] = knowledge_repr

        # [3] Knowledge Fusion
        if self.use_fusion and self.fusion_layer is not None:
            fused = self.fusion_layer(token_repr, knowledge_repr)
        else:
            fused = token_repr
        outputs["token_repr"] = fused

        # [4] Span Representation
        if self.use_span:
            span_repr, span_mask, span_indices = self.span_layer(fused, attention_mask)
            # [4b] Span Pruning
            if self.use_span_pruning and hasattr(self, "span_pruner"):
                span_repr, span_mask, span_indices, span_scores, top_indices = self.span_pruner(
                    span_repr, span_mask, span_indices)
                outputs["span_scores"] = span_scores
                outputs["pruning_top_indices"] = top_indices
            outputs["span_repr"] = span_repr
            outputs["span_mask"] = span_mask
            outputs["span_indices"] = span_indices

        # [5+6] Bi-Encoder Matching
        if self.use_biencoder and self.use_span:
            entity_type_repr = self.entity_type_encoder(device=token_repr.device)
            outputs["entity_type_repr"] = entity_type_repr
            matching_logits = self.span_entity_matcher(span_repr, entity_type_repr, span_mask)
            outputs["matching_logits"] = matching_logits
            raw_matching_logits = self.span_entity_matcher.compute_matching_logits(
                span_repr, entity_type_repr, span_mask)
            outputs["raw_matching_logits"] = raw_matching_logits
            # [6b] Evidence Aggregation
            span_token_logits = self.evidence_aggregator(
                matching_logits, span_indices, seq_len,
                span_logit_temperature=self.span_entity_matcher.span_logit_temperature)
            outputs["span_token_logits"] = span_token_logits

        # [7] KG Teacher passthrough
        if kg_teacher_embeddings is not None:
            outputs["kg_teacher_embeddings"] = kg_teacher_embeddings

        # [8] Multi-task Heads
        ner_logits = self.ner_head(fused)
        boundary_logits_out = self.boundary_head(fused)
        type_logits = self.type_head(fused)

        # [9] Gated Span-Token Fusion
        if self.use_biencoder and self.use_span and "span_token_logits" in outputs:
            if self.use_gated_fusion and hasattr(self, "span_token_gate"):
                ner_logits = self.span_token_gate(ner_logits, outputs["span_token_logits"])
            else:
                ner_logits = ner_logits + outputs["span_token_logits"]

        outputs["ner_logits"] = ner_logits
        outputs["boundary_logits"] = boundary_logits_out
        outputs["type_logits"] = type_logits

        # [10] CRF Decoder
        if self.use_crf:
            if labels is not None:
                crf_loss = self.crf(ner_logits, labels, mask=attention_mask)
                outputs["crf_loss"] = crf_loss
            with torch.no_grad():
                predictions = self.crf.decode(ner_logits, mask=attention_mask)
                outputs["predictions"] = predictions

        return outputs

    def get_predictions(self, input_ids, attention_mask, knowledge_embeddings, token_type_ids=None):
        self.eval()
        with torch.no_grad():
            outputs = self.forward(input_ids=input_ids, attention_mask=attention_mask,
                                   knowledge_embeddings=knowledge_embeddings, token_type_ids=token_type_ids)
        if self.use_crf:
            return outputs["predictions"]
        return outputs["ner_logits"].argmax(dim=-1).cpu().tolist()

    def count_parameters(self):
        counts = {
            "encoder": sum(p.numel() for p in self.encoder.parameters()),
            "knowledge_encoder": sum(p.numel() for p in self.knowledge_encoder.parameters()),
            "ner_head": sum(p.numel() for p in self.ner_head.parameters()),
            "total": sum(p.numel() for p in self.parameters()),
            "trainable": sum(p.numel() for p in self.parameters() if p.requires_grad),
        }
        return counts

# Alias for notebook usage
KoGNERv12 = KnowledgeAwareNERModelV11_6
log("Loaded: KoGNERv12 (KnowledgeAwareNERModelV11_6) — all components self-contained")

---
## 2b. Adaptive Loss Weighting (v13 NEW)

**Uncertainty-based multi-task weighting** (Kendall et al., CVPR 2018):
- Each task loss L_i is weighted by learned log-variance parameter `log_var_i`
- Weight = `1 / (2 * exp(log_var_i))`, with regularization term `log_var_i / 2`
- This automatically balances losses of different magnitudes

**Loss randomization**: At configurable intervals, non-essential losses are stochastically
dropped to test training stability and identify which losses are truly beneficial.

In [ ]:
# ============================================================================
# ADAPTIVE LOSS WEIGHTING (v13 NEW)
# Kendall et al., "Multi-Task Learning Using Uncertainty to Weigh Losses"
# ============================================================================

class AdaptiveLossWeighter(nn.Module):
    """Learns per-task uncertainty σ_i to automatically balance loss weights.

    L_total = Σ_i [ (1/(2σ²_i)) · L_i + (1/2) · log(σ²_i) ]

    Parameters are log(σ²_i) for numerical stability.
    """
    def __init__(self, loss_names, initial_log_var=0.0, min_weight=0.01, max_weight=10.0):
        super().__init__()
        self.loss_names = list(loss_names)
        self.num_tasks = len(self.loss_names)
        self.min_weight = min_weight
        self.max_weight = max_weight
        # Learnable log-variance per task: log(σ²)
        # Prefix keys with "lv_" to avoid nn.Module attribute conflicts
        self.log_vars = nn.ParameterDict({
            f"lv_{name}": nn.Parameter(torch.tensor(initial_log_var))
            for name in self.loss_names
        })

    def forward(self, loss_dict):
        """Apply adaptive weighting to a dictionary of losses.

        Args:
            loss_dict: dict of {loss_name: loss_tensor}

        Returns:
            weighted_total: adaptively weighted total loss
            weights: dict of effective weights per task
            uncertainties: dict of learned σ values per task
        """
        weighted_total = torch.tensor(0.0, device=next(iter(loss_dict.values())).device,
                                       requires_grad=True)
        weights = {}
        uncertainties = {}

        for name, loss_val in loss_dict.items():
            key = f"lv_{name}"
            if key in self.log_vars and loss_val.requires_grad:
                log_var = self.log_vars[key]
                # precision = 1/(2σ²) = 1/(2·exp(log_var))
                precision = 0.5 * torch.exp(-log_var)
                # Clamp for stability
                precision = precision.clamp(min=self.min_weight, max=self.max_weight)
                # Weighted loss + regularization
                weighted_loss = precision * loss_val + 0.5 * log_var
                weighted_total = weighted_total + weighted_loss
                weights[name] = precision.item()
                uncertainties[name] = torch.exp(0.5 * log_var).item()  # σ
            else:
                # Fallback: use loss as-is (for losses not in the adaptive set)
                weighted_total = weighted_total + loss_val
                weights[name] = 1.0

        return weighted_total, weights, uncertainties

    def get_current_weights(self):
        """Return current adaptive weights without computing losses."""
        weights = {}
        for name in self.loss_names:
            log_var = self.log_vars[f"lv_{name}"]
            precision = (0.5 * torch.exp(-log_var)).clamp(
                min=self.min_weight, max=self.max_weight
            )
            weights[name] = precision.item()
        return weights


class LossRandomizer:
    """Stochastically drops non-essential losses at configurable intervals.

    Used to analyze training stability: if dropping a loss causes instability,
    that loss is critical; if training is unaffected, the loss may be redundant.
    """
    def __init__(self, drop_prob=0.1, drop_interval=50,
                 exclude_from_drop=None):
        self.drop_prob = drop_prob
        self.drop_interval = drop_interval
        self.exclude = set(exclude_from_drop or ["ner", "crf"])
        self.step_count = 0
        self.drop_log = []  # track what was dropped and when

    def should_drop(self, loss_name):
        """Determine if a loss should be dropped at the current step."""
        self.step_count += 1
        if loss_name in self.exclude:
            return False
        if self.step_count % self.drop_interval != 0:
            return False
        drop = random.random() < self.drop_prob
        if drop:
            self.drop_log.append({"step": self.step_count, "dropped": loss_name})
        return drop

    def reset(self):
        self.step_count = 0

    def get_drop_stats(self):
        """Return summary of drops for analysis."""
        drops = Counter(d["dropped"] for d in self.drop_log)
        return dict(drops)

log("Loaded: AdaptiveLossWeighter, LossRandomizer")

---
## 2. Configuration, Labels & Synonym Augmenter

- 7-class BIO schema: O, B/I-GENE, B/I-DISEASE, B/I-CHEMICAL
- Entity type descriptions for bi-encoder
- Synonym dictionaries for minority-class augmentation

In [ ]:
# ============================================================================
# V12 CONFIG + LABELS + SYNONYMS + AUGMENTER
# ============================================================================
ENCODER_NAME = "dmis-lab/biobert-base-cased-v1.2"
MAX_LENGTH = 128

LABEL_LIST = ["O", "B-GENE", "I-GENE", "B-DISEASE", "I-DISEASE", "B-CHEMICAL", "I-CHEMICAL"]
LABEL2ID = {l: i for i, l in enumerate(LABEL_LIST)}
ID2LABEL = {i: l for l, i in LABEL2ID.items()}
NUM_LABELS = len(LABEL_LIST)

ENTITY_TYPE_MAP = {"O": 0, "GENE": 1, "DISEASE": 2, "CHEMICAL": 3}
NUM_ENTITY_TYPES = 4
BOUNDARY_MAP = {"O": 0, "B": 1, "I": 2}
NUM_BOUNDARY_CLASSES = 3
ENTITY_TYPES = ["GENE", "DISEASE", "CHEMICAL"]
MINORITY_TYPES = ["DISEASE", "CHEMICAL"]

DEFAULT_ENTITY_TYPE_DESCRIPTIONS = {
    "GENE": "gene protein DNA RNA enzyme kinase receptor transcription factor molecular biology",
    "DISEASE": "disease disorder syndrome illness condition pathology medical diagnosis clinical",
    "CHEMICAL": "chemical drug compound molecule medication pharmaceutical substance treatment therapy",
}

V8_BASELINE = {"overall_f1": 0.3798, "gene_f1": 0.5858, "disease_f1": 0.4452, "chemical_f1": 0.0254, "minority_f1": 0.1989}
V9_BASELINE = {"overall_f1": 0.4583, "gene_f1": 0.4930, "disease_f1": 0.1581, "chemical_f1": 0.0984, "minority_f1": 0.1333}

# ═══════════════════════════════════════════════════════════
# SYNONYM DICTIONARIES & AUGMENTER
# ═══════════════════════════════════════════════════════════
DISEASE_SYNONYMS = {
    "cancer": ["carcinoma", "malignancy", "tumor", "neoplasm"],
    "tumor": ["tumour", "neoplasm", "mass"],
    "diabetes": ["diabetic condition", "diabetes mellitus"],
    "hypertension": ["high blood pressure", "HTN"],
    "infection": ["infectious disease", "sepsis"],
    "inflammation": ["inflammatory condition", "swelling"],
    "fever": ["pyrexia", "hyperthermia"],
    "pain": ["ache", "discomfort"],
    "headache": ["cephalalgia", "migraine"],
    "nausea": ["queasiness", "sickness"],
    "fatigue": ["tiredness", "exhaustion", "lethargy"],
    "anemia": ["anaemia", "low hemoglobin"],
    "asthma": ["bronchial asthma"],
    "pneumonia": ["lung infection"],
    "hepatitis": ["liver inflammation"],
    "arthritis": ["joint inflammation"],
    "stroke": ["cerebrovascular accident", "CVA"],
    "depression": ["depressive disorder"],
    "edema": ["oedema", "fluid retention"],
    "bleeding": ["hemorrhage"],
    "thrombosis": ["blood clot"],
    "necrosis": ["tissue death"],
}
CHEMICAL_SYNONYMS = {
    "aspirin": ["acetylsalicylic acid", "ASA"],
    "ibuprofen": ["Advil", "Motrin", "NSAID"],
    "acetaminophen": ["paracetamol", "Tylenol"],
    "metformin": ["Glucophage", "biguanide"],
    "insulin": ["human insulin"],
    "warfarin": ["Coumadin"],
    "penicillin": ["antibiotic", "beta-lactam"],
    "ciprofloxacin": ["Cipro", "fluoroquinolone"],
    "methotrexate": ["MTX", "antimetabolite"],
    "prednisone": ["corticosteroid", "steroid"],
    "morphine": ["opioid"],
    "omeprazole": ["Prilosec", "PPI"],
    "atorvastatin": ["Lipitor", "statin"],
    "cisplatin": ["platinum compound", "chemotherapy"],
    "paclitaxel": ["Taxol", "taxane"],
    "rituximab": ["Rituxan", "monoclonal antibody"],
}

def build_reverse_mapping(sd):
    rev = {}
    for canon, syns in sd.items():
        rev[canon.lower()] = canon
        for s in syns: rev[s.lower()] = canon
    return rev

DISEASE_REVERSE = build_reverse_mapping(DISEASE_SYNONYMS)
CHEMICAL_REVERSE = build_reverse_mapping(CHEMICAL_SYNONYMS)

class SynonymAugmenter:
    def __init__(self, augment_prob=0.8, max_augments=5, seed=42):
        self.augment_prob = augment_prob
        self.max_augments = max_augments
        random.seed(seed)

    def _get_synonym(self, text, etype):
        t = text.lower()
        rev = DISEASE_REVERSE if etype == "DISEASE" else CHEMICAL_REVERSE
        sd = DISEASE_SYNONYMS if etype == "DISEASE" else CHEMICAL_SYNONYMS
        if t in rev:
            canon = rev[t]
            opts = [s for s in [canon] + sd.get(canon, []) if s.lower() != t]
            if opts: return random.choice(opts)
        return None

    def augment_dataset(self, data, target_ratio=0.3):
        minority = [ex for ex in data
                    if any(l.endswith("-DISEASE") or l.endswith("-CHEMICAL") for l in ex["labels"])]
        if len(minority) == 0:
            log("   Augmenter: no minority examples found, skipping augmentation")
            return data
        total_needed = max(0, int((target_ratio * len(data) - len(minority)) / (1 - target_ratio)))
        if total_needed == 0: return data
        log(f"   Augmenting: {len(minority)} minority, target {total_needed}")
        augmented = []
        random.shuffle(minority)
        for i in range(total_needed):
            ex = minority[i % len(minority)]
            augmented.append({"tokens": list(ex["tokens"]), "labels": list(ex["labels"]), "source": "aug"})
        log(f"   Added {len(augmented)} augmented examples")
        return data + augmented

log(f"v12 Config: {ENCODER_NAME}, {NUM_LABELS} labels, {ENTITY_TYPES}")
log(f"Synonyms: {len(DISEASE_SYNONYMS)} disease, {len(CHEMICAL_SYNONYMS)} chemical")

---
## 3. Data Loading

- **spyysalo/bc2gm_corpus** — gene/protein BIO tags (BC2GM corpus)
- **tner/bc5cdr** — chemical + disease BIO tags (BioCreative V CDR)

Both are standard Parquet datasets on HuggingFace (no `trust_remote_code` needed).

In [ ]:
# ============================================================================
# DATA LOADING (spyysalo/bc2gm_corpus + tner/bc5cdr)
# ============================================================================
PROGRESS.set_phase("DATA_LOAD")
tokenizer = AutoTokenizer.from_pretrained(ENCODER_NAME)

def load_bc2gm():
    """Load BC2GM gene corpus. Tags: 0=O, 1=B-GENE, 2=I-GENE."""
    log("Loading BC2GM (spyysalo/bc2gm_corpus)...")
    try:
        ds = load_dataset("spyysalo/bc2gm_corpus")
        log(f"  BC2GM splits: {list(ds.keys())}")
        return ds
    except Exception as e:
        log(f"  BC2GM load failed: {e}")
        return None

def load_bc5cdr():
    """Load BC5CDR chemical+disease corpus."""
    log("Loading BC5CDR (tner/bc5cdr)...")
    # Try multiple loading strategies (dataset script was deprecated)
    strategies = [
        ("parquet revision", lambda: load_dataset("tner/bc5cdr", revision="refs/convert/parquet")),
        ("trust_remote_code", lambda: load_dataset("tner/bc5cdr", trust_remote_code=True)),
        ("default", lambda: load_dataset("tner/bc5cdr")),
    ]
    for name, loader in strategies:
        try:
            ds = loader()
            log(f"  BC5CDR loaded via {name}, splits: {list(ds.keys())}")
            return ds
        except Exception as e:
            log(f"  BC5CDR {name} failed: {e}")
    log("  BC5CDR: all loading strategies failed!")
    return None

def tokenize_and_align(texts, labels_list):
    all_ids, all_mask, all_lab = [], [], []
    for toks, tags in zip(texts, labels_list):
        enc = tokenizer(toks, is_split_into_words=True, max_length=MAX_LENGTH,
                        padding="max_length", truncation=True)
        wids = enc.word_ids(); aligned = []; prev = None
        for wid in wids:
            if wid is None:
                aligned.append(-100)
            elif wid != prev:
                aligned.append(LABEL2ID.get(tags[wid], 0) if wid < len(tags) else -100)
            else:
                if wid < len(tags):
                    tag = tags[wid]
                    aligned.append(LABEL2ID.get("I-" + tag[2:], 0) if tag.startswith("B-") else LABEL2ID.get(tag, 0))
                else:
                    aligned.append(-100)
            prev = wid
        all_ids.append(enc["input_ids"]); all_mask.append(enc["attention_mask"]); all_lab.append(aligned)
    return all_ids, all_mask, all_lab

def prepare_data():
    log("Preparing data...")
    bc2gm = load_bc2gm()
    bc5cdr = load_bc5cdr()

    gene_texts, gene_labels = [], []
    chem_disease_texts, chem_disease_labels = [], []

    # --- BC2GM: Gene entities ---
    BC2GM_TAG_MAP = {0: "O", 1: "B-GENE", 2: "I-GENE"}
    if bc2gm:
        for split in ["train", "validation", "test"]:
            if split not in bc2gm: continue
            for ex in bc2gm[split]:
                tokens = ex.get("tokens", [])
                tags = ex.get("ner_tags", [])
                if not tokens or not tags: continue
                bio = [BC2GM_TAG_MAP.get(t, "O") for t in tags]
                if any(l != "O" for l in bio):
                    gene_texts.append(tokens)
                    gene_labels.append(bio)
        log(f"  BC2GM: {len(gene_texts)} examples with gene entities")
    else:
        log("  BC2GM not available!")

    # --- BC5CDR: Chemical + Disease entities ---
    BC5CDR_TAG_MAP = {0: "O", 1: "B-CHEMICAL", 4: "I-CHEMICAL", 2: "B-DISEASE", 3: "I-DISEASE"}
    if bc5cdr:
        for split in ["train", "validation", "test"]:
            if split not in bc5cdr: continue
            for ex in bc5cdr[split]:
                tokens = ex.get("tokens", [])
                tags = ex.get("tags", [])
                if not tokens or not tags: continue
                bio = [BC5CDR_TAG_MAP.get(t, "O") for t in tags]
                if any(l != "O" for l in bio):
                    chem_disease_texts.append(tokens)
                    chem_disease_labels.append(bio)
        log(f"  BC5CDR: {len(chem_disease_texts)} examples with chemical/disease entities")
    else:
        log("  BC5CDR not available!")

    if len(gene_texts) == 0 and len(chem_disease_texts) == 0:
        raise RuntimeError("No data loaded! Check dataset availability.")

    # ═══════════════════════════════════════════════════════════
    # BALANCED SAMPLING: cap gene examples to match chem/disease count
    # ═══════════════════════════════════════════════════════════
    n_cd = len(chem_disease_texts)
    n_gene = len(gene_texts)
    random.shuffle(gene_texts_labels := list(zip(gene_texts, gene_labels)))
    random.shuffle(chem_disease_texts_labels := list(zip(chem_disease_texts, chem_disease_labels)))

    max_gene = int(n_cd * 1.2) if n_cd > 0 else n_gene
    gene_texts_labels = gene_texts_labels[:max_gene]
    log(f"  Balanced: {len(gene_texts_labels)} gene (capped from {n_gene}), {len(chem_disease_texts_labels)} chem/disease")

    all_pairs = gene_texts_labels + chem_disease_texts_labels
    random.shuffle(all_pairs)

    # FIX 4: Apply synonym augmentation to boost minority class examples
    augmenter = SynonymAugmenter()
    all_data_for_aug = [{"tokens": t, "labels": l} for t, l in all_pairs]
    all_data_for_aug = augmenter.augment_dataset(all_data_for_aug, target_ratio=0.3)
    all_pairs = [(d["tokens"], d["labels"]) for d in all_data_for_aug]
    random.shuffle(all_pairs)

    all_texts = [p[0] for p in all_pairs]
    all_labels = [p[1] for p in all_pairs]
    log(f"Total examples (after augmentation): {len(all_texts)}")

    # Split: 70% train, 15% val, 15% test
    n = len(all_texts)
    te = int(n * 0.7); ve = int(n * 0.85)
    tr_t, tr_l = all_texts[:te], all_labels[:te]
    va_t, va_l = all_texts[te:ve], all_labels[te:ve]
    ts_t, ts_l = all_texts[ve:], all_labels[ve:]

    # Cap sizes for T4 memory
    tr_t, tr_l = tr_t[:6000], tr_l[:6000]
    va_t, va_l = va_t[:1500], va_l[:1500]
    ts_t, ts_l = ts_t[:2000], ts_l[:2000]
    log(f"Train:{len(tr_t)}, Val:{len(va_t)}, Test:{len(ts_t)}")

    # Log label distribution
    for split_name, split_labels in [("Train", tr_l), ("Val", va_l), ("Test", ts_l)]:
        entity_counts = Counter()
        for seq in split_labels:
            for l in seq:
                if l.startswith("B-"): entity_counts[l[2:]] += 1
        log(f"  {split_name} entity distribution: {dict(entity_counts)}")

    # Tokenize
    tri, trm, trl = tokenize_and_align(tr_t, tr_l)
    vai, vam, val_ = tokenize_and_align(va_t, va_l)
    tsi, tsm, tsl = tokenize_and_align(ts_t, ts_l)
    PROGRESS.complete_phase("DATA_LOAD")
    return {"train": {"input_ids": tri, "attention_mask": trm, "labels": trl},
            "val":   {"input_ids": vai, "attention_mask": vam, "labels": val_},
            "test":  {"input_ids": tsi, "attention_mask": tsm, "labels": tsl}}

data = prepare_data()

---
## 4. Knowledge Store (FAISS)

- 45 curated biomedical knowledge entries (15 GENE, 15 DISEASE, 15 CHEMICAL)
- Encode with BioBERT CLS → FAISS index for nearest-neighbor retrieval
- Results cached to Drive for fast resume

In [ ]:
# ============================================================================
# KNOWLEDGE STORE (FAISS + 45 curated entries) — with Drive caching
# ============================================================================
PROGRESS.set_phase("KNOWLEDGE_BUILD")

if "tokenizer" not in dir() or tokenizer is None:
    tokenizer = AutoTokenizer.from_pretrained(ENCODER_NAME)
if "data" not in dir() or data is None:
    data = prepare_data()

# Clear stale cache from previous runs
_cache_path = os.path.join(CKPT.ckpt_dir, "data_cache.pkl")
if os.path.exists(_cache_path):
    os.remove(_cache_path)
    log("Cleared stale data cache")
os.makedirs(CKPT.ckpt_dir, exist_ok=True)
for _f in os.listdir(CKPT.ckpt_dir):
    _fp = os.path.join(CKPT.ckpt_dir, _f)
    if os.path.isfile(_fp): os.remove(_fp)
log("Cleared old checkpoints")

KNOWLEDGE_ENTRIES = [
    ("TP53 tumor protein p53 transcription factor regulates cell cycle", "GENE"),
    ("BRCA1 breast cancer gene DNA repair genome stability", "GENE"),
    ("EGFR epidermal growth factor receptor tyrosine kinase", "GENE"),
    ("KRAS oncogene GTPase cell proliferation pathways", "GENE"),
    ("MYC proto-oncogene transcription factor cell growth", "GENE"),
    ("AKT1 serine threonine kinase PI3K signaling", "GENE"),
    ("VEGF vascular endothelial growth factor angiogenesis", "GENE"),
    ("BCL2 anti-apoptotic protein programmed cell death", "GENE"),
    ("HER2 epidermal growth factor receptor 2 breast cancer", "GENE"),
    ("PTEN phosphatase tensin homolog tumor suppressor", "GENE"),
    ("ALK anaplastic lymphoma kinase fusion gene", "GENE"),
    ("BRAF serine threonine kinase MAPK signaling", "GENE"),
    ("CDK4 cyclin dependent kinase 4 cell cycle", "GENE"),
    ("ERBB2 receptor tyrosine kinase 2 cancers", "GENE"),
    ("JAK2 janus kinase 2 cytokine receptor signaling", "GENE"),
    ("Alzheimer disease progressive neurodegenerative dementia", "DISEASE"),
    ("Type 2 diabetes mellitus metabolic insulin resistance", "DISEASE"),
    ("Breast cancer malignant neoplasm breast tissue", "DISEASE"),
    ("Parkinson disease neurodegenerative movement disorder", "DISEASE"),
    ("COPD chronic obstructive pulmonary disease lung", "DISEASE"),
    ("Rheumatoid arthritis autoimmune inflammatory joint", "DISEASE"),
    ("Multiple sclerosis demyelinating central nervous system", "DISEASE"),
    ("Acute myeloid leukemia cancer myeloid blood cells", "DISEASE"),
    ("Lupus erythematosus autoimmune connective tissue", "DISEASE"),
    ("Non-small cell lung cancer common lung cancer", "DISEASE"),
    ("Crohn disease chronic inflammatory bowel GI tract", "DISEASE"),
    ("Glioblastoma aggressive malignant brain tumor", "DISEASE"),
    ("Cystic fibrosis genetic disorder lungs pancreas", "DISEASE"),
    ("Myocardial infarction heart attack coronary occlusion", "DISEASE"),
    ("Hepatocellular carcinoma primary liver malignancy", "DISEASE"),
    ("Metformin biguanide antidiabetic type 2 diabetes", "CHEMICAL"),
    ("Ibuprofen NSAID anti-inflammatory pain relief", "CHEMICAL"),
    ("Cisplatin platinum chemotherapy cancer treatment", "CHEMICAL"),
    ("Doxorubicin anthracycline chemotherapy various cancers", "CHEMICAL"),
    ("Methotrexate antifolate cancer autoimmune diseases", "CHEMICAL"),
    ("Aspirin acetylsalicylic acid antiplatelet anti-inflammatory", "CHEMICAL"),
    ("Tamoxifen estrogen receptor modulator breast cancer", "CHEMICAL"),
    ("Rituximab monoclonal antibody CD20 lymphoma", "CHEMICAL"),
    ("Paclitaxel taxane chemotherapy microtubules", "CHEMICAL"),
    ("Erlotinib EGFR tyrosine kinase inhibitor lung cancer", "CHEMICAL"),
    ("Omeprazole proton pump inhibitor gastric acid", "CHEMICAL"),
    ("Warfarin anticoagulant blood clot prevention", "CHEMICAL"),
    ("Statins HMG-CoA reductase inhibitors cholesterol", "CHEMICAL"),
    ("Insulin peptide hormone blood glucose metabolism", "CHEMICAL"),
    ("Bevacizumab anti-VEGF monoclonal antibody angiogenesis", "CHEMICAL"),
]
log(f"Knowledge entries: {len(KNOWLEDGE_ENTRIES)}")

log("Encoding knowledge entries...")
_enc_model = AutoModel.from_pretrained(ENCODER_NAME).to(DEVICE); _enc_model.eval()
entry_embeddings, entry_types = [], []
with torch.no_grad():
    for text, etype in KNOWLEDGE_ENTRIES:
        enc = tokenizer(text, return_tensors="pt", max_length=64, padding="max_length", truncation=True).to(DEVICE)
        cls = _enc_model(**enc).last_hidden_state[:, 0, :].cpu().numpy()
        entry_embeddings.append(cls[0]); entry_types.append(etype)
del _enc_model; gc.collect(); torch.cuda.empty_cache()
entry_embeddings = np.array(entry_embeddings, dtype=np.float32)

if HAS_FAISS:
    dim = entry_embeddings.shape[1]; faiss_index = faiss.IndexFlatIP(dim)
    normed = entry_embeddings.copy(); faiss.normalize_L2(normed); faiss_index.add(normed)
    log(f"FAISS index: {faiss_index.ntotal} entries, dim={dim}")
else:
    faiss_index = None; normed = entry_embeddings

def retrieve_knowledge(input_ids_list, top_k=5):
    em = AutoModel.from_pretrained(ENCODER_NAME).to(DEVICE); em.eval()
    all_k = []
    all_indices = []
    with torch.no_grad():
        for ids in input_ids_list:
            it = torch.tensor([ids], device=DEVICE)
            mk = (it != tokenizer.pad_token_id).long()
            cls = em(input_ids=it, attention_mask=mk).last_hidden_state[:, 0, :].cpu().numpy()
            if faiss_index:
                q = cls.copy(); faiss.normalize_L2(q)
                _, idx = faiss_index.search(q, top_k)
                ret = normed[idx[0]]
                all_indices.append(idx[0].copy())
            else:
                q = cls[0]; q = q / (np.linalg.norm(q) + 1e-8)
                sims = normed @ q
                top_idx = np.argsort(-sims)[:top_k]
                ret = normed[top_idx]
                all_indices.append(top_idx.copy())
            all_k.append(ret)
    del em; gc.collect(); torch.cuda.empty_cache()
    return np.array(all_k, dtype=np.float32), np.array(all_indices, dtype=np.int64)

log("Retrieving knowledge...")
train_knowledge, train_ret_indices = retrieve_knowledge(data["train"]["input_ids"])
val_knowledge, val_ret_indices = retrieve_knowledge(data["val"]["input_ids"])
test_knowledge, test_ret_indices = retrieve_knowledge(data["test"]["input_ids"])
log(f"Knowledge: train={train_knowledge.shape}, val={val_knowledge.shape}, test={test_knowledge.shape}")

CKPT.save_data(data, train_knowledge, val_knowledge, test_knowledge)
PROGRESS.complete_phase("KNOWLEDGE_BUILD")

---
## 5. v13 Configuration

Key changes from v12:
- **lambda_ner**: 2.0 → **1.0** (rebalanced)
- **Data**: 5000 → **1500** train samples (fast iteration)
- **Batch size**: 16 → **32** (A100)
- **Adaptive loss**: enabled (uncertainty-based)
- **Loss randomization**: enabled (stability testing)

In [ ]:
# ============================================================================
# v13 CONFIGURATION — Fast Iteration + Adaptive Loss
# ============================================================================

def get_v13_config():
    """v13 config: reduced data, adaptive loss, A100 optimized."""
    return {
        "model": {
            "encoder_name": "dmis-lab/biobert-base-cased-v1.2",
            "max_length": 128,
            "hidden_dim": 768,
            "dropout_rate": 0.2,       # ↓ from 0.3 (v12)
            "freeze_encoder": False,
            "num_labels": 7,
            "span_logit_temperature": 1.5,
            "max_spans_per_sequence": 100,
            "span_aggregation_method": "logsumexp",
        },
        "span_representation": {
            "enabled": True, "max_span_width": 8, "span_hidden_dim": 768,
            "span_ffn_dim": 1024, "use_width_embedding": True,
            "max_width_embedding": 8, "dropout": 0.1, "pooling": "endpoint",
        },
        "knowledge_fusion": {
            "method": "attention", "hidden_dim": 768,
            "attention": {"num_heads": 4, "dropout": 0.1,
                          "use_gated_residual": True, "residual_gate_dim": 256},
            "distillation": {"enabled": True, "distill_dim": 256, "temperature": 2.0},
        },
        "biencoder": {
            "enabled": True, "use_pretrained_encoder": True,
            "freeze_type_encoder": True, "ffn_dim": 1024, "temperature": 1.0,
            "entity_type_descriptions": {
                "GENE": "gene protein DNA RNA enzyme kinase receptor transcription factor molecular biology",
                "DISEASE": "disease disorder syndrome illness condition pathology medical diagnosis clinical",
                "CHEMICAL": "chemical drug compound molecule medication pharmaceutical substance treatment therapy",
            },
        },
        "kg_teacher": {
            "enabled": True, "num_relations": 4, "spatial_dim": 256,
            "logical_dim": 256, "gnn_layers": 2, "gnn_heads": 4,
        },
        "crf": {"enabled": True, "num_labels": 7, "use_hard_constraints": True},
        # ── v13 KEY CHANGE: lambda_ner reduced to 1.0 ──
        "losses": {
            "lambda_ner": 1.0,             # ↓ from 2.0 [v13]
            "lambda_language": 0.5,
            "lambda_boundary": 0.3,
            "lambda_type": 0.3,
            "lambda_align": 0.1,
            "lambda_distillation": 0.05,
            "lambda_contrastive": 0.0,     # disabled
            "bce_pos_weight": 5.0,
            "alignment_margin": 0.2,
        },
        "features": {
            "use_focal_loss": True,
            "use_cross_attention": True,
            "use_knowledge_fusion": True,
            "use_span_representation": True,
            "use_biencoder": True,
            "use_language_loss": True,
            "use_distillation_loss": True,
            "use_crf_decoder": True,
            "use_contrastive_loss": False,  # disabled in v13
            "use_gated_span_fusion": True,
            "use_span_pruning": True,
            "fusion_method": "attention",
            "use_adaptive_loss": True,      # v13 NEW
        },
        "training": {
            "batch_size": 32,              # ↑ from 16 (A100)
            "learning_rate": 2e-5,         # ↑ from 1e-5
            "weight_decay": 0.01,
            "max_grad_norm": 1.0,
            "gradient_accumulation_steps": 1,  # ↓ from 2 (A100)
            "patience": 10,
            "log_interval": 25,
        },
        "training_schedule": {
            "stage1_epochs": 4,
            "stage2a_epochs": 5,
            "stage2b_epochs": 8,
        },
        # ── v13: Reduced data for fast experimentation ──
        "data": {
            "max_train_samples": 1500,     # ↓ from 5000 (70% reduction)
            "max_val_samples": 400,        # ↓ from 1000
            "max_test_samples": 500,       # ↓ from 2000
            "max_few_nerd_samples": 10000, # ↓ from 50000
            "use_bc2gm": True,
            "use_synonym_augmentation": True,
            "augment_prob": 0.8,
            "max_augments_per_example": 3, # ↓ from 5
        },
        "weighted_sampler": {
            "use_weighted_sampler": True,
            "minority_oversample_weight": 5.0,
        },
        # ── v13 NEW: Adaptive loss config ──
        "adaptive_loss": {
            "enabled": True,
            "initial_log_var": 0.0,
            "min_weight": 0.01,
            "max_weight": 10.0,
            "randomization": {
                "enabled": True,
                "drop_prob": 0.1,
                "drop_interval": 50,
                "exclude_from_drop": ["ner", "crf"],
            },
        },
    }

# Ablation configs
def get_v13_ablation_configs():
    base = get_v13_config()
    ablations = {}

    # v13_baseline — no knowledge, no CRF
    abl = copy.deepcopy(base)
    for key in ["use_span_representation", "use_biencoder", "use_language_loss",
                "use_knowledge_fusion", "use_crf_decoder", "use_contrastive_loss",
                "use_distillation_loss"]:
        abl["features"][key] = False
    abl["features"]["use_adaptive_loss"] = False
    abl["features"]["fusion_method"] = "none"
    ablations["v13_baseline"] = abl

    # v13_full — full pipeline
    ablations["v13_full"] = copy.deepcopy(base)

    return ablations

base_config = get_v13_config()
log(f"v13 Config loaded: lambda_ner={base_config['losses']['lambda_ner']}, "
    f"train_samples={base_config['data']['max_train_samples']}, "
    f"batch_size={base_config['training']['batch_size']}, "
    f"adaptive_loss={base_config['features']['use_adaptive_loss']}")

---
## 5. KG Teacher Pretraining

Build a biomedical knowledge graph and pretrain the KG Teacher module:
1. Build KG: same-type edges (`related_to`) + cross-type keyword overlap (`interacts_with`)
2. Pretrain GNN on node classification (GENE/DISEASE/CHEMICAL)
3. Pretrain TransR on relational triples
4. Generate frozen teacher embeddings `H = [Z, Z', Z'']`

In [ ]:
# ============================================================================
# KG TEACHER PRETRAINING (Component [7] in .tex architecture)
# ============================================================================
PROGRESS.set_phase("KG_TEACHER_PRETRAIN")

def build_kg_teacher():
    """
    Build biomedical KG and pretrain KG Teacher.
    Returns H = [Z, Z', Z''] frozen teacher embeddings.
    """
    log("=" * 60)
    log("  KG TEACHER PRETRAINING")
    log("=" * 60)

    num_entities = len(KNOWLEDGE_ENTRIES)
    TYPE_LABEL_MAP = {"GENE": 1, "DISEASE": 2, "CHEMICAL": 3}
    node_labels_list = [TYPE_LABEL_MAP[etype] for _, etype in KNOWLEDGE_ENTRIES]
    node_labels = torch.tensor(node_labels_list, dtype=torch.long, device=DEVICE)

    # Extract keywords for cross-type edge detection
    STOP_WORDS = {"with", "that", "from", "this", "into", "most", "common",
                  "type", "used", "based", "line", "cell", "cells"}
    entry_keywords = []
    for text, _ in KNOWLEDGE_ENTRIES:
        words = set(w for w in text.lower().split() if len(w) > 3 and w not in STOP_WORDS)
        entry_keywords.append(words)

    # Build adjacency + triples
    adjacency = torch.zeros(num_entities, num_entities, device=DEVICE)
    triples = []
    for i in range(num_entities):
        for j in range(i + 1, num_entities):
            type_i, type_j = KNOWLEDGE_ENTRIES[i][1], KNOWLEDGE_ENTRIES[j][1]
            if type_i == type_j:
                adjacency[i, j] = adjacency[j, i] = 1.0
                triples.extend([(i, 1, j), (j, 1, i)])
            else:
                overlap = entry_keywords[i] & entry_keywords[j]
                if len(overlap) >= 2:
                    adjacency[i, j] = adjacency[j, i] = 1.0
                    triples.extend([(i, 3, j), (j, 3, i)])
    for i in range(num_entities):
        adjacency[i, i] = 1.0

    num_edges = int((adjacency.sum().item() - num_entities) / 2)
    log(f"  KG built: {num_entities} nodes, {num_edges} edges, {len(triples)} triples")

    # Instantiate KGTeacher
    kg_teacher = KGTeacher(
        num_entities=num_entities, num_relations=4,
        text_dim=768, spatial_dim=256, logical_dim=256,
        gnn_layers=2, gnn_heads=4, num_entity_classes=4, dropout=0.1,
    ).to(DEVICE)

    # Encode entry texts with BioBERT
    log("  Encoding entry texts with BioBERT...")
    encoder_model = AutoModel.from_pretrained(ENCODER_NAME).to(DEVICE)
    encoder_model.eval()
    text_embeddings_list = []
    with torch.no_grad():
        for text, _ in KNOWLEDGE_ENTRIES:
            enc = tokenizer(text, return_tensors="pt", max_length=64,
                          padding="max_length", truncation=True).to(DEVICE)
            cls_emb = encoder_model(**enc).last_hidden_state[:, 0, :]
            text_embeddings_list.append(cls_emb.squeeze(0))
    text_embeddings_Z = torch.stack(text_embeddings_list, dim=0)
    log(f"  Text embeddings Z: {text_embeddings_Z.shape}")
    del encoder_model; gc.collect(); torch.cuda.empty_cache()

    # Pretrain GNN
    log("  Pretraining GNN (node classification)...")
    gnn_result = kg_teacher.pretrain_gnn(
        text_embeddings=text_embeddings_Z, adjacency=adjacency,
        node_labels=node_labels, epochs=50, lr=1e-3,
    )
    log(f"  GNN pretrain done: final_loss={gnn_result['final_loss']:.4f}")

    # Pretrain TransR
    log("  Pretraining TransR (relational embeddings)...")
    if len(triples) > 0:
        transr_result = kg_teacher.pretrain_transr(
            triples=triples, epochs=100, lr=1e-3, negative_samples=5,
        )
        log(f"  TransR pretrain done: final_loss={transr_result['final_loss']:.4f}")
    else:
        log("  No triples found, skipping TransR pretraining")

    # Generate H = [Z, Z', Z'']
    log("  Generating combined KG teacher embeddings H = [Z, Z', Z'']...")
    kg_teacher.eval()
    with torch.no_grad():
        H = kg_teacher(text_embeddings_Z, adjacency)
    log(f"  KG Teacher H: {H.shape} (dim={H.shape[1]})")
    log("  KG Teacher pretraining complete.")
    log("=" * 60)
    return H

kg_teacher_H = build_kg_teacher()
log(f"KG Teacher ready: H shape={kg_teacher_H.shape}")
PROGRESS.complete_phase("KG_TEACHER_PRETRAIN")

---
## 7b. Training Visualization Dashboard (v13 NEW)

Colab-compatible visualization system using matplotlib/seaborn:
- **Loss curves**: Training & validation loss on same plot
- **F1 scores**: Overall + class-based (GENE, DISEASE, CHEMICAL) per epoch
- **Stability analysis**: Detect loss spikes, gradient explosions
- **Adaptive weights**: Track how uncertainty-based weights evolve
- **Stage boundaries**: Vertical lines marking training stage transitions

All plots update after each epoch and are saved to Google Drive.

In [ ]:
# ============================================================================
# TRAINING VISUALIZATION DASHBOARD (v13 NEW)
# Colab-compatible: matplotlib + seaborn
# ============================================================================

class TrainingVisualizer:
    """Generates comprehensive training plots for Colab."""

    STAGE_COLORS = {
        "stage1_warmup": "#3498db",
        "stage2a_structural": "#e74c3c",
        "stage2b_contrastive": "#2ecc71",
    }

    def __init__(self, save_dir=None):
        self.save_dir = save_dir or (DRIVE_BASE if DRIVE_AVAILABLE else LOCAL_BASE)
        os.makedirs(os.path.join(self.save_dir, "plots"), exist_ok=True)
        sns.set_style("whitegrid")
        plt.rcParams.update({"figure.dpi": 100, "font.size": 11})

    def _add_stage_boundaries(self, ax, history):
        """Add vertical lines at training stage transitions."""
        if not history.stages:
            return
        prev_stage = history.stages[0]
        for i, stage in enumerate(history.stages):
            if stage != prev_stage:
                ax.axvline(x=history.epochs[i], color="gray",
                          linestyle="--", alpha=0.5, linewidth=0.8)
                ax.text(history.epochs[i], ax.get_ylim()[1] * 0.95,
                       stage.replace("_", "\n"), fontsize=7, ha="center",
                       color="gray", alpha=0.7)
                prev_stage = stage

    def _detect_spikes(self, values, threshold=2.0):
        """Detect sudden jumps in a metric series."""
        spikes = []
        for i in range(1, len(values)):
            if values[i-1] > 0 and values[i] / values[i-1] > threshold:
                spikes.append(i)
        return spikes

    def plot_loss_curves(self, history):
        """Plot training loss with component breakdown."""
        fig, axes = plt.subplots(1, 2, figsize=(14, 5))

        # Left: Total training loss
        ax = axes[0]
        ax.plot(history.epochs, history.train_loss, "b-o", markersize=4,
                label="Train Loss", linewidth=2)
        spikes = self._detect_spikes(history.train_loss)
        for s in spikes:
            ax.annotate("⚠ spike", xy=(history.epochs[s], history.train_loss[s]),
                        fontsize=8, color="red", fontweight="bold")
        self._add_stage_boundaries(ax, history)
        ax.set_xlabel("Epoch"); ax.set_ylabel("Loss")
        ax.set_title("Training Loss per Epoch")
        ax.legend(); ax.grid(True, alpha=0.3)

        # Right: Loss components
        ax = axes[1]
        if history.loss_components:
            all_keys = set()
            for lc in history.loss_components:
                all_keys.update(lc.keys())
            for key in sorted(all_keys):
                vals = [lc.get(key, 0) for lc in history.loss_components]
                ax.plot(history.epochs, vals, "-o", markersize=3, label=key)
        ax.set_xlabel("Epoch"); ax.set_ylabel("Loss Value")
        ax.set_title("Loss Components per Epoch")
        ax.legend(fontsize=8, ncol=2); ax.grid(True, alpha=0.3)

        plt.tight_layout()
        self._save_fig(fig, "loss_curves")
        plt.show()

    def plot_f1_scores(self, history):
        """Plot F1 scores: overall + class-based on same axes."""
        fig, axes = plt.subplots(1, 2, figsize=(14, 5))

        # Left: Overall & Minority F1
        ax = axes[0]
        ax.plot(history.epochs, history.overall_f1, "b-o", markersize=4,
                label="Overall F1", linewidth=2)
        ax.plot(history.epochs, history.minority_f1, "r-s", markersize=4,
                label="Minority F1", linewidth=2)
        self._add_stage_boundaries(ax, history)
        ax.set_xlabel("Epoch"); ax.set_ylabel("F1 Score")
        ax.set_title("Overall & Minority F1 per Epoch")
        ax.set_ylim(-0.02, 1.02)
        ax.legend(); ax.grid(True, alpha=0.3)

        # Right: Per-class F1
        ax = axes[1]
        ax.plot(history.epochs, history.gene_f1, "g-^", markersize=4,
                label="GENE F1", linewidth=2)
        ax.plot(history.epochs, history.disease_f1, "r-v", markersize=4,
                label="DISEASE F1", linewidth=2)
        ax.plot(history.epochs, history.chemical_f1, "m-D", markersize=4,
                label="CHEMICAL F1", linewidth=2)
        self._add_stage_boundaries(ax, history)
        ax.set_xlabel("Epoch"); ax.set_ylabel("F1 Score")
        ax.set_title("Class-Based F1 per Epoch (GENE, DISEASE, CHEMICAL)")
        ax.set_ylim(-0.02, 1.02)
        ax.legend(); ax.grid(True, alpha=0.3)

        plt.tight_layout()
        self._save_fig(fig, "f1_scores")
        plt.show()

    def plot_adaptive_weights(self, history):
        """Plot evolution of adaptive loss weights over training."""
        if not history.adaptive_weights or not any(history.adaptive_weights):
            return
        fig, ax = plt.subplots(figsize=(10, 5))
        all_keys = set()
        for aw in history.adaptive_weights:
            all_keys.update(aw.keys())
        for key in sorted(all_keys):
            vals = [aw.get(key, 1.0) for aw in history.adaptive_weights]
            ax.plot(history.epochs, vals, "-o", markersize=3, label=f"w({key})")
        ax.set_xlabel("Epoch"); ax.set_ylabel("Adaptive Weight (1/2σ²)")
        ax.set_title("Adaptive Loss Weights Evolution (Uncertainty-Based)")
        ax.legend(fontsize=8, ncol=2); ax.grid(True, alpha=0.3)
        plt.tight_layout()
        self._save_fig(fig, "adaptive_weights")
        plt.show()

    def plot_gradient_norms(self, history):
        """Plot gradient norms per module to detect vanishing/exploding gradients."""
        if not history.gradient_norms or not any(history.gradient_norms):
            return
        fig, ax = plt.subplots(figsize=(10, 5))
        all_keys = set()
        for gn in history.gradient_norms:
            all_keys.update(gn.keys())
        for key in sorted(all_keys):
            vals = [gn.get(key, 0) for gn in history.gradient_norms]
            ax.plot(history.epochs, vals, "-o", markersize=3, label=key)
        ax.set_xlabel("Epoch"); ax.set_ylabel("Gradient Norm")
        ax.set_title("Per-Module Gradient Norms (Stability Analysis)")
        ax.legend(fontsize=8, ncol=2); ax.grid(True, alpha=0.3)
        # Mark explosion threshold
        ax.axhline(y=100.0, color="red", linestyle="--", alpha=0.5, label="Explosion threshold")
        plt.tight_layout()
        self._save_fig(fig, "gradient_norms")
        plt.show()

    def plot_stability_summary(self, history):
        """Combined stability dashboard: loss + F1 + overfitting detection."""
        fig, axes = plt.subplots(2, 2, figsize=(14, 10))

        # (0,0): Train loss with spike detection
        ax = axes[0][0]
        ax.plot(history.epochs, history.train_loss, "b-o", markersize=4, linewidth=2)
        spikes = self._detect_spikes(history.train_loss)
        for s in spikes:
            ax.plot(history.epochs[s], history.train_loss[s], "rv", markersize=12)
        ax.set_title(f"Train Loss (⚠ {len(spikes)} spikes detected)")
        ax.set_xlabel("Epoch"); ax.set_ylabel("Loss"); ax.grid(True, alpha=0.3)

        # (0,1): All F1 scores together
        ax = axes[0][1]
        ax.plot(history.epochs, history.overall_f1, "b-", linewidth=2, label="Overall")
        ax.plot(history.epochs, history.gene_f1, "g--", linewidth=1.5, label="GENE")
        ax.plot(history.epochs, history.disease_f1, "r--", linewidth=1.5, label="DISEASE")
        ax.plot(history.epochs, history.chemical_f1, "m--", linewidth=1.5, label="CHEMICAL")
        ax.set_title("F1 Scores (All Classes)")
        ax.set_xlabel("Epoch"); ax.set_ylabel("F1"); ax.legend(fontsize=8)
        ax.set_ylim(-0.02, 1.02); ax.grid(True, alpha=0.3)

        # (1,0): Learning rate schedule
        ax = axes[1][0]
        if history.lr_history:
            ax.plot(history.epochs, history.lr_history, "k-", linewidth=2)
        self._add_stage_boundaries(ax, history)
        ax.set_title("Learning Rate Schedule")
        ax.set_xlabel("Epoch"); ax.set_ylabel("LR"); ax.grid(True, alpha=0.3)

        # (1,1): Overfitting detection — train loss vs F1 divergence
        ax = axes[1][1]
        if len(history.train_loss) > 1:
            # Normalize train loss to [0,1] for comparison
            tl = np.array(history.train_loss)
            tl_norm = (tl - tl.min()) / (tl.max() - tl.min() + 1e-8)
            ax.plot(history.epochs, 1 - tl_norm, "b-", label="1 - norm(train_loss)", linewidth=2)
            ax.plot(history.epochs, history.overall_f1, "r-", label="Val F1", linewidth=2)
            # Overfitting = train improves but val doesn't
            gap = (1 - tl_norm) - np.array(history.overall_f1)
            ax.fill_between(history.epochs, 0, np.maximum(gap, 0),
                           alpha=0.2, color="red", label="Overfitting gap")
        ax.set_title("Overfitting Detection")
        ax.set_xlabel("Epoch"); ax.set_ylabel("Score"); ax.legend(fontsize=8)
        ax.set_ylim(-0.1, 1.1); ax.grid(True, alpha=0.3)

        plt.suptitle("v13 Training Stability Dashboard", fontsize=14, fontweight="bold")
        plt.tight_layout()
        self._save_fig(fig, "stability_summary")
        plt.show()

    def plot_all(self, history):
        """Generate all visualization plots."""
        if not history.epochs:
            log("No training history to visualize yet.")
            return
        log(f"Generating plots for {len(history.epochs)} epochs...")
        self.plot_loss_curves(history)
        self.plot_f1_scores(history)
        self.plot_adaptive_weights(history)
        self.plot_gradient_norms(history)
        self.plot_stability_summary(history)
        log("All plots generated.")

    def _save_fig(self, fig, name):
        path = os.path.join(self.save_dir, "plots", f"{name}.png")
        try:
            fig.savefig(path, bbox_inches="tight", dpi=150)
        except Exception as e:
            log(f"Warning: Could not save plot {name}: {e}")

VIZ = TrainingVisualizer()
log("Loaded: TrainingVisualizer (Colab-compatible dashboard)")

---
## 8. Training Loop (v13 — Adaptive Loss + Visualization)

Modified from v12:
- **Adaptive loss weighting** via `AdaptiveLossWeighter` (Kendall et al.)
- **Loss randomization** at configurable intervals
- **TrainingHistory** records all metrics for visualization
- **bf16 autocast** on A100
- **Per-epoch visualization** via `TrainingVisualizer`
- **Stability detection**: flags loss spikes and gradient explosions

In [ ]:
# train_v13 is defined after evaluate_model (cell below).
# This cell left as no-op to avoid duplicate definitions.
log('train_v13 defined in next training cell')


In [ ]:
# ============================================================================
# DATASET & DATALOADER
# ============================================================================
PROGRESS.set_phase("MODEL_DEFINE")

class NERDataset(Dataset):
    def __init__(self, encodings, labels, knowledge_embs, top_k=5, emb_dim=768,
                 kg_retrieval_indices=None):
        self.encodings = encodings
        self.labels = labels
        self.knowledge = knowledge_embs
        self.top_k = top_k
        self.emb_dim = emb_dim
        self.kg_retrieval_indices = kg_retrieval_indices

    def __len__(self):
        return len(self.labels)

    def __getitem__(self, idx):
        item = {
            "input_ids": torch.tensor(self.encodings["input_ids"][idx]),
            "attention_mask": torch.tensor(self.encodings["attention_mask"][idx]),
            "labels": torch.tensor(self.labels[idx], dtype=torch.long),
        }
        ner_labels = item["labels"]
        boundary = torch.full_like(ner_labels, -100)
        type_labels = torch.full_like(ner_labels, -100)
        for i in range(len(ner_labels)):
            lid = ner_labels[i].item()
            if lid == -100: continue
            label_name = ID2LABEL.get(lid, "O")
            boundary[i] = BOUNDARY_MAP.get(label_name[0] if label_name != "O" else "O", 0)
            if label_name == "O":
                type_labels[i] = 0
            else:
                etype = label_name[2:] if len(label_name) > 2 else "O"
                type_labels[i] = ENTITY_TYPE_MAP.get(etype, 0)
        item["boundary_labels"] = boundary
        item["type_labels"] = type_labels

        if self.knowledge is not None and idx < len(self.knowledge):
            item["knowledge_embeddings"] = torch.tensor(self.knowledge[idx], dtype=torch.float32)
        else:
            item["knowledge_embeddings"] = torch.zeros(self.top_k, self.emb_dim)

        if self.kg_retrieval_indices is not None and idx < len(self.kg_retrieval_indices):
            item["kg_retrieval_indices"] = torch.tensor(self.kg_retrieval_indices[idx], dtype=torch.long)
        else:
            item["kg_retrieval_indices"] = torch.zeros(self.top_k, dtype=torch.long)
        return item

# Create datasets
train_dataset = NERDataset(
    data["train"], data["train"]["labels"], train_knowledge,
    top_k=5, emb_dim=768, kg_retrieval_indices=train_ret_indices,
)
val_dataset = NERDataset(
    data["val"], data["val"]["labels"], val_knowledge,
    top_k=5, emb_dim=768, kg_retrieval_indices=val_ret_indices,
)
test_dataset = NERDataset(
    data["test"], data["test"]["labels"], test_knowledge,
    top_k=5, emb_dim=768, kg_retrieval_indices=test_ret_indices,
)

# WeightedRandomSampler: oversample minority examples
# TASK_11: Reduced minority weight from 20.0 to 5.0 to prevent overfitting
minority_label_ids = {LABEL2ID.get("B-DISEASE"), LABEL2ID.get("I-DISEASE"),
                      LABEL2ID.get("B-CHEMICAL"), LABEL2ID.get("I-CHEMICAL")}
minority_label_ids.discard(None)
sample_weights = []
for label_seq in data["train"]["labels"]:
    has_minority = any(lid in minority_label_ids for lid in label_seq if lid != -100)
    sample_weights.append(5.0 if has_minority else 1.0)  # TASK_11: reduced from 20.0
sampler = WeightedRandomSampler(weights=sample_weights, num_samples=len(sample_weights), replacement=True)

train_loader = DataLoader(train_dataset, batch_size=16, sampler=sampler, num_workers=0)
val_loader = DataLoader(val_dataset, batch_size=32, shuffle=False, num_workers=0)
test_loader = DataLoader(test_dataset, batch_size=32, shuffle=False, num_workers=0)

log(f"Datasets: train={len(train_dataset)}, val={len(val_dataset)}, test={len(test_dataset)}")
log(f"TASK_11 applied: Minority oversampling reduced from 20x to 5x")
PROGRESS.complete_phase("MODEL_DEFINE")

---
## 7. Training & Evaluation Functions

### Training: 3-Stage Schedule (v11.5, matches .tex)
- **Stage 1** (3 ep): NER + CRF + language (BCE) — encoder warmup
- **Stage 2A** (5 ep): + boundary + type + distillation + alignment
- **Stage 2B** (7 ep): + contrastive + alignment (drops distillation)

### Evaluation
- Entity-level micro F1 (overall + per-type + minority)

In [ ]:
# ============================================================================
# EVALUATION FUNCTION
# ============================================================================

def _extract_entities(label_ids):
    """Extract entity spans as (start, end) tuples grouped by type."""
    entities = defaultdict(list)
    current_type = None
    current_start = None
    for i, lid in enumerate(label_ids):
        if lid == -100:
            if current_type:
                entities[current_type].append((current_start, i))
                current_type = None
            continue
        label = ID2LABEL.get(lid, "O")
        if label.startswith("B-"):
            if current_type:
                entities[current_type].append((current_start, i))
            current_type = label[2:]
            current_start = i
        elif label.startswith("I-"):
            tag_type = label[2:]
            if current_type != tag_type:
                if current_type:
                    entities[current_type].append((current_start, i))
                current_type = tag_type
                current_start = i
        else:
            if current_type:
                entities[current_type].append((current_start, i))
                current_type = None
    if current_type:
        entities[current_type].append((current_start, len(label_ids)))
    return dict(entities)


def evaluate_model(model, data_loader):
    """Evaluate model and return entity-level metrics."""
    model.eval()
    all_preds, all_labels = [], []
    
    # TASK_2: Track prediction label distribution
    prediction_counter = Counter()

    with torch.no_grad():
        for batch in data_loader:
            input_ids = batch["input_ids"].to(DEVICE)
            attention_mask = batch["attention_mask"].to(DEVICE)
            labels = batch["labels"]
            knowledge = batch["knowledge_embeddings"].to(DEVICE)

            outputs = model(
                input_ids=input_ids,
                attention_mask=attention_mask,
                knowledge_embeddings=knowledge,
            )
            if "predictions" in outputs:
                preds = outputs["predictions"]
            else:
                preds = outputs["ner_logits"].argmax(dim=-1).cpu().tolist()

            for pred_seq, label_seq in zip(preds, labels.tolist()):
                all_preds.append(pred_seq)
                all_labels.append(label_seq)
                
                # TASK_2: Count predictions (excluding padding)
                for p, l in zip(pred_seq, label_seq):
                    if l != -100:
                        prediction_counter[p] += 1

    # TASK_2: Log prediction distribution
    total_preds = sum(prediction_counter.values())
    pred_dist_str = " | ".join([
        f"{ID2LABEL.get(lid, f'ID{lid}')}={count/total_preds*100:.1f}%"
        for lid, count in sorted(prediction_counter.items())
    ])
    log(f"   Prediction distribution: {pred_dist_str}")
    
    # Check for all-O collapse
    o_label_id = LABEL2ID.get("O", 0)
    o_percentage = prediction_counter.get(o_label_id, 0) / max(total_preds, 1) * 100
    if o_percentage > 95.0:
        log(f"   ⚠️  WARNING: Model predicting mostly O tags ({o_percentage:.1f}%)")

    # Entity-level evaluation
    tp, fp, fn = defaultdict(int), defaultdict(int), defaultdict(int)
    for preds, labels in zip(all_preds, all_labels):
        pred_entities = _extract_entities(preds)
        gold_entities = _extract_entities(labels)
        for et in ENTITY_TYPES:
            pred_set = set(pred_entities.get(et, []))
            gold_set = set(gold_entities.get(et, []))
            tp[et] += len(pred_set & gold_set)
            fp[et] += len(pred_set - gold_set)
            fn[et] += len(gold_set - pred_set)

    per_type = {}
    for et in ENTITY_TYPES:
        p = tp[et] / (tp[et] + fp[et]) if (tp[et] + fp[et]) > 0 else 0.0
        r = tp[et] / (tp[et] + fn[et]) if (tp[et] + fn[et]) > 0 else 0.0
        f1 = 2 * p * r / (p + r) if (p + r) > 0 else 0.0
        per_type[et] = {"precision": p, "recall": r, "f1": f1}

    total_tp = sum(tp.values()); total_fp = sum(fp.values()); total_fn = sum(fn.values())
    overall_p = total_tp / (total_tp + total_fp) if (total_tp + total_fp) > 0 else 0.0
    overall_r = total_tp / (total_tp + total_fn) if (total_tp + total_fn) > 0 else 0.0
    overall_f1 = 2 * overall_p * overall_r / (overall_p + overall_r) if (overall_p + overall_r) > 0 else 0.0

    min_tp = sum(tp[et] for et in MINORITY_TYPES)
    min_fp = sum(fp[et] for et in MINORITY_TYPES)
    min_fn = sum(fn[et] for et in MINORITY_TYPES)
    min_p = min_tp / (min_tp + min_fp) if (min_tp + min_fp) > 0 else 0.0
    min_r = min_tp / (min_tp + min_fn) if (min_tp + min_fn) > 0 else 0.0
    minority_f1 = 2 * min_p * min_r / (min_p + min_r) if (min_p + min_r) > 0 else 0.0

    return {
        "overall_micro_f1": overall_f1,
        "minority_micro_f1": minority_f1,
        "gene_f1": per_type["GENE"]["f1"],
        "disease_f1": per_type["DISEASE"]["f1"],
        "chemical_f1": per_type["CHEMICAL"]["f1"],
        "per_type": per_type,
    }

log("Loaded: evaluate_model (with prediction distribution tracking)")

In [ ]:
# ============================================================================
# TRAINING FUNCTION — v13 with adaptive loss, bf16, visualization
# ============================================================================

def train_v13(model, train_loader, val_loader, config, experiment_name="v13",
              kg_teacher_H_tensor=None):
    global HISTORY

    base_lr = config.get("training", {}).get("learning_rate", 1e-5)
    max_grad_norm = config.get("training", {}).get("max_grad_norm", 1.0)
    log_interval = config.get("training", {}).get("log_interval", 50)
    accumulation_steps = config.get("training", {}).get("gradient_accumulation_steps", 1)
    patience = config.get("training", {}).get("patience", 10)

    features = config.get("features", {})
    losses_cfg = config.get("losses", {})
    losses_cfg["use_focal_loss"] = features.get("use_focal_loss", True)
    losses_cfg["use_language_loss"] = features.get("use_language_loss", False)
    losses_cfg["use_contrastive_loss"] = features.get("use_contrastive_loss", False)
    losses_cfg["use_distillation_loss"] = features.get("use_distillation_loss", False)
    losses_cfg["hidden_dim"] = config.get("model", {}).get("hidden_dim", 768)

    label_counts = Counter()
    for batch in train_loader:
        labels = batch["labels"].reshape(-1)
        for l in labels.tolist():
            if l != -100: label_counts[l] += 1
    total_count = sum(label_counts.values())
    focal_alpha = []
    for i in range(NUM_LABELS):
        count = label_counts.get(i, 1)
        ratio = count / total_count
        alpha = min(5.0, max(0.1, 1.0 / (ratio * NUM_LABELS)))
        focal_alpha.append(alpha)
    losses_cfg["focal_alpha"] = focal_alpha
    log(f"Focal alpha: {[f'{a:.2f}' for a in focal_alpha]}")

    criterion = CompositeLossV12(losses_cfg).to(DEVICE)

    # v13: Adaptive Loss Weighter
    adaptive_cfg = config.get("adaptive_loss", {})
    use_adaptive = adaptive_cfg.get("enabled", False)
    adaptive_weighter = None
    if use_adaptive:
        loss_names = ["ner", "crf", "language", "boundary", "type",
                      "distillation", "alignment"]
        adaptive_weighter = AdaptiveLossWeighter(
            loss_names=loss_names,
            initial_log_var=adaptive_cfg.get("initial_log_var", 0.0),
            min_weight=adaptive_cfg.get("min_weight", 0.01),
            max_weight=adaptive_cfg.get("max_weight", 10.0),
        ).to(DEVICE)
        log(f"Adaptive loss weighting ENABLED ({len(loss_names)} losses)")

    # v13: Loss Randomizer
    rand_cfg = config.get("loss_randomization", {})
    use_randomizer = rand_cfg.get("enabled", False)
    loss_randomizer = None
    if use_randomizer:
        loss_randomizer = LossRandomizer(
            drop_prob=rand_cfg.get("drop_prob", 0.1),
            drop_interval=rand_cfg.get("randomize_every_n_steps", 50),
            exclude_from_drop=rand_cfg.get("protected_losses", ["ner", "crf"]),
        )
        log(f"Loss randomization ENABLED (drop_prob={rand_cfg.get('drop_prob', 0.1)})")

    # BUGFIX-1: TrainingSchedule expects a config dict, not keyword args
    schedule = TrainingSchedule(config)
    total_epochs = schedule.total_epochs

    param_groups = [{"params": model.parameters(), "lr": base_lr}]
    if adaptive_weighter is not None:
        adaptive_lr = base_lr * adaptive_cfg.get("lr_multiplier", 0.01)
        param_groups.append({"params": adaptive_weighter.parameters(), "lr": adaptive_lr})

    optimizer = AdamW(param_groups, weight_decay=config.get("training", {}).get("weight_decay", 0.01))
    max_lrs = [base_lr]
    if adaptive_weighter is not None:
        max_lrs.append(base_lr * adaptive_cfg.get("lr_multiplier", 0.01))
    steps_per_epoch = len(train_loader) // accumulation_steps
    total_steps = total_epochs * steps_per_epoch + 10
    scheduler = OneCycleLR(optimizer, max_lr=max_lrs, total_steps=max(total_steps, 1),
                           pct_start=0.1, anneal_strategy="cos")

    start_epoch = 0
    best_minority_f1 = 0.0
    best_overall_f1 = 0.0
    best_state = None
    best_epoch = 0
    patience_counter = 0

    ckpt = CKPT.load_training_checkpoint(experiment_name)
    if ckpt is not None:
        model.load_state_dict(ckpt["model_state_dict"])
        optimizer.load_state_dict(ckpt["optimizer_state_dict"])
        try:
            scheduler.load_state_dict(ckpt["scheduler_state_dict"])
        except Exception:
            pass
        start_epoch = ckpt["epoch"] + 1
        best_minority_f1 = ckpt.get("best_f1", 0.0)
        best_state = ckpt.get("best_state", None)
        log(f"Resumed from epoch {start_epoch}")

    amp_dtype = torch.bfloat16 if USE_BF16 else torch.float32
    use_amp = USE_BF16

    log(f"Training {experiment_name}: {total_epochs} ep, lr={base_lr}, "
        f"bs={config['training']['batch_size']}, accum={accumulation_steps}, bf16={use_amp}")

    PROGRESS.set_phase(f"TRAIN_{experiment_name}")
    start_time = time.time()

    for global_epoch in range(start_epoch + 1, total_epochs + 1):
        stage = schedule.get_stage_for_epoch(global_epoch)
        active_losses = set(stage.active_losses)
        model.train()
        if adaptive_weighter:
            adaptive_weighter.train()

        epoch_loss = 0.0
        epoch_loss_components = defaultdict(float)
        epoch_grad_norms = []
        num_batches = 0
        global_step = (global_epoch - 1) * len(train_loader)

        for batch_idx, batch in enumerate(train_loader):
            input_ids = batch["input_ids"].to(DEVICE)
            attention_mask = batch["attention_mask"].to(DEVICE)
            labels = batch["labels"].to(DEVICE)
            # BUGFIX-5: Use dataset-provided boundary/type labels
            boundary_labels = batch["boundary_labels"].to(DEVICE)
            type_labels = batch["type_labels"].to(DEVICE)
            knowledge = batch["knowledge_embeddings"].to(DEVICE)

            # BUGFIX-4: Apply loss randomization BEFORE criterion call
            use_boundary = "boundary" in active_losses
            use_type = "type" in active_losses
            use_alignment = "alignment" in active_losses
            use_distill = "distillation" in active_losses

            if loss_randomizer is not None:
                if loss_randomizer.should_drop("boundary"): use_boundary = False
                if loss_randomizer.should_drop("type"): use_type = False
                if loss_randomizer.should_drop("alignment"): use_alignment = False
                if loss_randomizer.should_drop("distillation"): use_distill = False

            with torch.autocast(device_type="cuda", dtype=amp_dtype, enabled=use_amp):
                outputs = model(input_ids=input_ids, attention_mask=attention_mask,
                                knowledge_embeddings=knowledge, labels=labels)

                loss_kwargs = {"ner_logits": outputs["ner_logits"], "ner_labels": labels,
                               "attention_mask": attention_mask}

                if "boundary_logits" in outputs and use_boundary:
                    loss_kwargs["boundary_logits"] = outputs["boundary_logits"]
                    loss_kwargs["boundary_labels"] = boundary_labels

                if "type_logits" in outputs and use_type:
                    loss_kwargs["type_logits"] = outputs["type_logits"]
                    loss_kwargs["type_labels"] = type_labels

                if outputs.get("token_repr") is not None:
                    loss_kwargs["token_repr"] = outputs["token_repr"]
                if outputs.get("knowledge_repr") is not None and use_alignment:
                    loss_kwargs["knowledge_repr"] = outputs["knowledge_repr"]
                if "crf_loss" in outputs and "crf" in active_losses:
                    loss_kwargs["crf_loss"] = outputs["crf_loss"]
                if "matching_logits" in outputs and "language" in active_losses:
                    loss_kwargs["matching_logits"] = outputs.get("matching_logits")
                    stl = outputs.get("span_type_labels")
                    if stl is None and "span_indices" in outputs:
                        stl = create_span_type_labels(labels, outputs["span_indices"])
                    loss_kwargs["span_type_labels"] = stl
                    loss_kwargs["span_mask"] = outputs.get("span_mask")
                if outputs.get("span_repr") is not None:
                    loss_kwargs["span_repr"] = outputs["span_repr"]
                if kg_teacher_H_tensor is not None and use_distill:
                    loss_kwargs["kg_teacher_embeddings"] = kg_teacher_H_tensor

                loss_dict = criterion(**loss_kwargs)

                # BUGFIX-2: adaptive_weighter returns (total, weights, uncertainties)
                if adaptive_weighter is not None:
                    raw = {k: v for k, v in loss_dict.items()
                           if k != "total" and isinstance(v, torch.Tensor) and v.requires_grad}
                    if raw:
                        total_loss, _, _ = adaptive_weighter(raw)
                    else:
                        total_loss = loss_dict["total"]
                else:
                    total_loss = loss_dict["total"]

                total_loss = total_loss / accumulation_steps

            total_loss.backward()

            if (batch_idx + 1) % accumulation_steps == 0:
                gn = nn.utils.clip_grad_norm_(model.parameters(), max_grad_norm).item()
                epoch_grad_norms.append(gn)
                optimizer.step(); scheduler.step(); optimizer.zero_grad()

            epoch_loss += total_loss.item() * accumulation_steps
            for k, v in loss_dict.items():
                if k != "total" and isinstance(v, torch.Tensor):
                    epoch_loss_components[k] += v.item()
            num_batches += 1

            if (batch_idx + 1) % log_interval == 0:
                log(f"  [{experiment_name}] Ep{global_epoch} batch {batch_idx+1}/{len(train_loader)} "
                    f"loss={epoch_loss/num_batches:.4f} stage={stage.name}")

        avg_loss = epoch_loss / max(num_batches, 1)
        avg_comps = {k: v / max(num_batches, 1) for k, v in epoch_loss_components.items()}
        avg_gn = float(np.mean(epoch_grad_norms)) if epoch_grad_norms else 0.0
        # BUGFIX-3: method is get_current_weights(), not get_weights()
        adaptive_wts = adaptive_weighter.get_current_weights() if adaptive_weighter else {}

        val_metrics = evaluate_model(model, val_loader)
        of1 = val_metrics.get("overall_micro_f1", 0)
        mf1 = val_metrics.get("minority_micro_f1", 0)
        cur_lr = scheduler.get_last_lr()[0] if hasattr(scheduler, 'get_last_lr') else base_lr

        log(f"  Ep{global_epoch}/{total_epochs} [{stage.name}] loss={avg_loss:.4f} "
            f"F1={of1:.4f} minF1={mf1:.4f} gn={avg_gn:.2f}")
        if adaptive_wts:
            log(f"  Adaptive: {' '.join(f'{k}={v:.3f}' for k,v in adaptive_wts.items())}")

        HISTORY.record_epoch(epoch=global_epoch, stage=stage.name, train_loss=avg_loss,
            val_metrics=val_metrics, loss_comps=avg_comps,
            grad_norms={"mean": avg_gn}, adaptive_wts=adaptive_wts, lr=cur_lr)

        if len(HISTORY.epochs) >= 2:
            try:
                VIZ.plot_all(HISTORY)
            except Exception as e:
                log(f"  Viz error: {e}")

        PROGRESS.update_training(epoch=global_epoch, stage=stage.name,
            loss=avg_loss, overall_f1=of1, minority_f1=mf1)

        if mf1 > best_minority_f1:
            best_minority_f1 = mf1; best_overall_f1 = of1
            best_state = copy.deepcopy(model.state_dict())
            best_epoch = global_epoch; patience_counter = 0
            log(f"   New best minority F1: {best_minority_f1:.4f}")
        else:
            patience_counter += 1
            if patience_counter >= patience:
                log(f"   Early stopping at epoch {global_epoch}")
                break

        CKPT.save_training_checkpoint(experiment_name, global_epoch - 1, model,
            optimizer, scheduler, best_minority_f1, best_state, HISTORY.to_dict())

    if best_state is not None:
        model.load_state_dict(best_state)
        log(f"Restored best model epoch {best_epoch} (minF1={best_minority_f1:.4f})")

    if loss_randomizer:
        drop_stats = loss_randomizer.get_drop_stats()
        log(f"Loss randomization stats: {drop_stats}")

    total_time = time.time() - start_time
    log(f"Training done in {total_time/60:.1f} min")
    return {"best_epoch": best_epoch, "best_minority_f1": best_minority_f1,
            "best_overall_f1": best_overall_f1, "total_time_min": total_time / 60}

log("Loaded: train_v13 (adaptive loss + bf16 + visualization)")


---
## 8. Ablation Sweep

Five progressive experiments testing each architectural component:

| # | Experiment | Components |
|---|-----------|------------|
| 1 | `v13_baseline` | BioBERT + token classification only |
| 2 | `v13_knowledge_concat` | + concat fusion + bi-encoder + KG distill |
| 3 | `v13_knowledge_gated` | + gated fusion |
| 4 | `v13_knowledge_attention` | + attention fusion + contrastive + distill |
| 5 | `v13_knowledge_crf` | Full: attention + CRF + all losses |

In [ ]:
# ============================================================================
# ABLATION CONFIGS + SWEEP + COMPARISON
# ============================================================================

ABLATION_CONFIGS = {
    "v13_baseline": {
        "description": "BioBERT + token classification only (no KoGNER, no extensions)",
        "features": {
            "use_span_representation": False,
            "use_biencoder": False,
            "use_language_loss": False,
            "use_knowledge_fusion": False,
            "use_crf_decoder": False,
            "use_contrastive_loss": False,
            "use_distillation_loss": False,
            "use_focal_loss": True,
            "use_calibration": True,
            "fusion_method": "none",
            "use_gated_span_fusion": False,
            "use_span_pruning": False,
            "span_aggregation_method": "logsumexp",
            "span_logit_temperature": 1.5,
            "max_spans_per_sequence": 100,
        },
    },
    "v13_knowledge_concat": {
        "description": "KoGNER bi-encoder + span + BCE + concat fusion + KG distill",
        "features": {
            "use_span_representation": True,
            "use_biencoder": True,
            "use_language_loss": True,
            "use_knowledge_fusion": True,
            "use_crf_decoder": False,
            "use_contrastive_loss": False,
            "use_distillation_loss": True,
            "use_focal_loss": True,
            "use_calibration": True,
            "fusion_method": "concat",
            "use_gated_span_fusion": True,
            "use_span_pruning": True,
            "span_aggregation_method": "logsumexp",
            "span_logit_temperature": 1.5,
            "max_spans_per_sequence": 100,
        },
    },
    "v13_knowledge_gated": {
        "description": "KoGNER bi-encoder + span + BCE + gated fusion + KG distill",
        "features": {
            "use_span_representation": True,
            "use_biencoder": True,
            "use_language_loss": True,
            "use_knowledge_fusion": True,
            "use_crf_decoder": False,
            "use_contrastive_loss": False,
            "use_distillation_loss": True,
            "use_focal_loss": True,
            "use_calibration": True,
            "fusion_method": "gated",
            "use_gated_span_fusion": True,
            "use_span_pruning": True,
            "span_aggregation_method": "logsumexp",
            "span_logit_temperature": 1.5,
            "max_spans_per_sequence": 100,
        },
    },
    "v13_knowledge_attention": {
        "description": "KoGNER bi-encoder + span + BCE + attention + KG distill + contrastive",
        "features": {
            "use_span_representation": True,
            "use_biencoder": True,
            "use_language_loss": True,
            "use_knowledge_fusion": True,
            "use_crf_decoder": False,
            "use_contrastive_loss": True,
            "use_distillation_loss": True,
            "use_focal_loss": True,
            "use_calibration": True,
            "fusion_method": "attention",
            "use_gated_span_fusion": True,
            "use_span_pruning": True,
            "span_aggregation_method": "logsumexp",
            "span_logit_temperature": 1.5,
            "max_spans_per_sequence": 100,
        },
    },
    "v13_knowledge_crf": {
        "description": "Full: KoGNER bi-encoder + BCE + attention + CRF + KG distill + contrastive",
        "features": {
            "use_span_representation": True,
            "use_biencoder": True,
            "use_language_loss": True,
            "use_knowledge_fusion": True,
            "use_crf_decoder": True,
            "use_contrastive_loss": True,
            "use_distillation_loss": True,
            "use_focal_loss": True,
            "use_calibration": True,
            "fusion_method": "attention",
            "use_gated_span_fusion": True,
            "use_span_pruning": True,
            "span_aggregation_method": "logsumexp",
            "span_logit_temperature": 1.5,
            "max_spans_per_sequence": 100,
        },
    },
}


def run_ablation_sweep(train_loader, val_loader, test_loader, base_config,
                       kg_teacher_H_tensor=None):
    """Run all v13 ablation experiments."""
    PROGRESS.set_phase("ABLATION_SWEEP")
    log(f"\n{'='*60}")
    log(f"  RUNNING v13 ABLATION SWEEP ({len(ABLATION_CONFIGS)} experiments)")
    log(f"{'='*60}")

    # Check for previously completed experiments
    prev_state = CKPT.load_ablation_state()
    completed_set = set(prev_state["completed"]) if prev_state else set()
    results = prev_state.get("results", {}) if prev_state else {}

    for idx, (name, abl_cfg) in enumerate(ABLATION_CONFIGS.items()):
        if name in completed_set:
            log(f"\n  Skipping {name} (already completed)")
            continue

        log(f"\n  Ablation {idx+1}/{len(ABLATION_CONFIGS)}: {name}")
        log(f"  {abl_cfg['description']}")
        PROGRESS.update_ablation(current=name, desc=abl_cfg["description"],
                                 index=idx+1, total=len(ABLATION_CONFIGS))

        config = copy.deepcopy(base_config)
        config["features"] = abl_cfg["features"]
        config["knowledge_fusion"]["method"] = abl_cfg["features"].get("fusion_method", "attention")

        try:
            # Phase 1: Model construction
            log(f"  [{name}] Building model...")
            model = KoGNERv12(config).to(DEVICE)

            # Phase 2: Pre-flight — one forward + loss pass
            log(f"  [{name}] Pre-flight check...")
            model.train()
            test_batch = next(iter(train_loader))
            with torch.no_grad():
                _input_ids = test_batch["input_ids"].to(DEVICE)
                _attn_mask = test_batch["attention_mask"].to(DEVICE)
                _labels = test_batch["labels"].to(DEVICE)
                _knowledge = test_batch["knowledge_embeddings"].to(DEVICE)
                _outputs = model(input_ids=_input_ids, attention_mask=_attn_mask,
                                 knowledge_embeddings=_knowledge, labels=_labels)
                log(f"  [{name}] Forward OK — keys: {list(_outputs.keys())}")
                # Quick loss check
                features = config.get("features", {})
                _losses_cfg = copy.deepcopy(config.get("losses", {}))
                _losses_cfg["use_focal_loss"] = features.get("use_focal_loss", True)
                _losses_cfg["use_language_loss"] = features.get("use_language_loss", False)
                _losses_cfg["use_contrastive_loss"] = features.get("use_contrastive_loss", False)
                _losses_cfg["use_distillation_loss"] = features.get("use_distillation_loss", False)
                _losses_cfg["hidden_dim"] = config.get("model", {}).get("hidden_dim", 768)
                _losses_cfg["focal_alpha"] = [1.0] * config.get("model", {}).get("num_labels", 7)
                _criterion = CompositeLossV12(_losses_cfg).to(DEVICE)
                _loss_dict = _criterion(
                    ner_logits=_outputs["ner_logits"], ner_labels=_labels,
                    crf_loss=_outputs.get("crf_loss"),
                    attention_mask=_attn_mask,
                )
                log(f"  [{name}] Loss OK — {', '.join(f'{k}={v.item():.4f}' for k,v in _loss_dict.items())}")
                del _criterion, _loss_dict, _outputs
            log(f"  [{name}] Pre-flight PASSED")

            # Phase 3: Training
            log(f"  [{name}] Starting training...")
            train_result = train_v13(
                model, train_loader, val_loader, config,
                experiment_name=name,
                kg_teacher_H_tensor=kg_teacher_H_tensor,
            )

            # Phase 4: Final evaluation
            log(f"  [{name}] Final evaluation...")
            test_metrics = evaluate_model(model, test_loader)

            results[name] = {"train": train_result, "test": test_metrics, "config": abl_cfg}
            log(f"  {name}: Overall={test_metrics['overall_micro_f1']:.4f}, "
                f"Minority={test_metrics['minority_micro_f1']:.4f}")

            PROGRESS.update_results(name, {
                "overall_f1": test_metrics["overall_micro_f1"],
                "minority_f1": test_metrics["minority_micro_f1"],
                "gene_f1": test_metrics["gene_f1"],
                "disease_f1": test_metrics["disease_f1"],
                "chemical_f1": test_metrics["chemical_f1"],
            })

            # Save best model
            CKPT.save_best_model(name, model.state_dict(), test_metrics)

        except Exception as e:
            import traceback
            err_tb = traceback.format_exc()
            log(f"\n  *** {name} FAILED ***")
            log(f"  Error: {e}")
            log(f"  Traceback (last 5 lines):")
            for line in err_tb.strip().split("\n")[-5:]:
                log(f"    {line}")
            PROGRESS.log_error(f"Ablation {name}: {e}")
            results[name] = {"error": str(e), "traceback": err_tb[-500:]}

        completed_set.add(name)
        CKPT.save_ablation_state(completed_set, results)

        try:
            del model
        except NameError:
            pass
        gc.collect(); torch.cuda.empty_cache()

    PROGRESS.complete_phase("ABLATION_SWEEP")
    return results


def print_comparison(ablation_results, baselines=None):
    """Print comparison table of all experiments."""
    log(f"\n{'='*80}")
    log(f"  v13 EXPERIMENT COMPARISON")
    log(f"{'='*80}")
    log(f"  {'Experiment':<30} {'Overall':>8} {'Minority':>8} {'GENE':>8} {'DISEASE':>8} {'CHEM':>8}")
    log(f"  {'-'*30} {'-'*8} {'-'*8} {'-'*8} {'-'*8} {'-'*8}")

    if baselines:
        for name, vals in baselines.items():
            if vals.get("overall_f1") is not None:
                log(f"  {name:<30} {vals['overall_f1']:>8.4f} {vals.get('minority_f1',0):>8.4f} "
                    f"{vals.get('gene_f1',0):>8.4f} {vals.get('disease_f1',0):>8.4f} "
                    f"{vals.get('chemical_f1',0):>8.4f}")
    log(f"  {'-'*30} {'-'*8} {'-'*8} {'-'*8} {'-'*8} {'-'*8}")

    for name, result in ablation_results.items():
        if "error" in result:
            err_msg = result["error"][:80]
            log(f"  {name:<30} {'ERROR':>8}")
            log(f"    -> {err_msg}")
            continue
        test = result.get("test", {})
        log(f"  {name:<30} {test.get('overall_micro_f1',0):>8.4f} "
            f"{test.get('minority_micro_f1',0):>8.4f} "
            f"{test.get('gene_f1',0):>8.4f} "
            f"{test.get('disease_f1',0):>8.4f} "
            f"{test.get('chemical_f1',0):>8.4f}")

log("Ablation sweep functions defined")

---
## 9. Run Full v13 Pipeline

Executes the complete pipeline:
1. Build base config with loss lambdas matching `.tex` architecture
2. Run 5-experiment ablation sweep
3. Print comparison table with v8/v9 baselines
4. Save all results to Google Drive

In [ ]:
# ============================================================================
# BASE CONFIG — v13 (lambda_ner=1.0, batch=32, adaptive loss)
# ============================================================================
base_config = {
    "model": {
        "encoder_name": ENCODER_NAME, "hidden_dim": 768, "max_length": MAX_LENGTH,
        "dropout_rate": 0.2, "freeze_encoder": False, "num_labels": NUM_LABELS,
        "num_entity_types": 4, "num_boundary_classes": 3,
        "max_spans_per_sequence": 100, "span_logit_temperature": 1.5,
        "span_aggregation_method": "logsumexp",
    },
    "span_representation": {
        "enabled": True, "max_span_width": 8, "span_ffn_dim": 1024,
        "use_width_embedding": True, "dropout": 0.1, "pooling": "endpoint",
    },
    "knowledge_store": {"embedding_dim": 768, "retrieval": {"top_k": 5}},
    "knowledge_encoder": {"mode": "broadcast"},
    "knowledge_fusion": {
        "method": "attention",
        "concat": {"dropout": 0.1},
        "gated": {"gate_hidden_dim": 256, "dropout": 0.1},
        "attention": {"num_heads": 4, "dropout": 0.1,
                      "use_gated_residual": True, "residual_gate_dim": 256},
    },
    "crf": {"enabled": True, "num_labels": NUM_LABELS, "use_hard_constraints": True},
    "features": {
        "use_span_representation": True, "use_biencoder": True,
        "use_language_loss": True, "use_knowledge_fusion": True,
        "use_crf_decoder": True, "use_contrastive_loss": False,
        "use_distillation_loss": True, "use_focal_loss": True,
        "use_calibration": True, "fusion_method": "attention",
        "use_gated_span_fusion": True, "use_span_pruning": True,
        "span_aggregation_method": "logsumexp", "span_logit_temperature": 1.5,
        "max_spans_per_sequence": 100,
    },
    "losses": {
        "lambda_ner": 1.0,
        "lambda_language": 0.5, "lambda_boundary": 0.3, "lambda_type": 0.3,
        "lambda_align": 0.1, "lambda_contrastive": 0.2, "lambda_distillation": 0.05,
        "alignment_margin": 0.2, "contrastive_temperature": 0.07, "kg_teacher_dim": 1280,
    },
    "training": {
        "batch_size": 32, "learning_rate": 1e-5, "weight_decay": 0.01,
        "max_grad_norm": 1.0, "log_interval": 50,
        "gradient_accumulation_steps": 1, "patience": 10,
    },
    "training_schedule": {"stage1_epochs": 4, "stage2a_epochs": 5, "stage2b_epochs": 8},
    "adaptive_loss": {
        "enabled": True, "initial_log_var": 0.0, "lr_multiplier": 0.01,
        "min_weight": 0.01, "max_weight": 10.0,
    },
    "loss_randomization": {
        "enabled": True, "drop_prob": 0.1,
        "protected_losses": ["ner", "crf"], "randomize_every_n_steps": 50,
    },
}

CKPT.save_ablation_state(set(), {})
log("v13 config: lambda_ner=1.0, bs=32, adaptive=ON, bf16=" + str(USE_BF16))

ablation_results = run_ablation_sweep(
    train_loader, val_loader, test_loader, base_config,
    kg_teacher_H_tensor=kg_teacher_H,
)

baselines = {
    "v8_baseline":  {"overall_f1": 0.3798, "minority_f1": 0.1989},
    "v9.1_baseline": {"overall_f1": 0.4583, "minority_f1": 0.1333},
}
print_comparison(ablation_results, baselines)

PROGRESS.set_phase("FINAL_SAVE")
serializable = {}
for name, result in ablation_results.items():
    if "error" in result:
        serializable[name] = result
    else:
        serializable[name] = {
            "train": result.get("train", {}),
            "test": {k: float(v) if isinstance(v, (float, np.floating)) else v
                    for k, v in result.get("test", {}).items() if k != "per_type"},
        }
results_dir = os.path.join(CKPT.base_dir, "final_results")
os.makedirs(results_dir, exist_ok=True)
with open(os.path.join(results_dir, "ablation_results_v13.json"), "w") as f:
    json.dump(serializable, f, indent=2)
CKPT.save_final_results(serializable)
CKPT.save_history(HISTORY)
PROGRESS.set_complete()
log("v13 Pipeline Complete")
VIZ.plot_all(HISTORY)


---
## 10. Post-Training Visualization

Generate all plots from training history. Can be re-run anytime to regenerate
plots from saved history (even after Colab disconnection).

In [ ]:
# ============================================================================
# POST-TRAINING VISUALIZATION — Regenerate all plots
# ============================================================================

# Attempt to load saved history (resilience after disconnect)
saved_history = CKPT.load_history()
if saved_history is not None:
    HISTORY = saved_history
    log(f"Loaded saved history: {len(HISTORY.epochs)} epochs")

# Generate all dashboard plots
VIZ.plot_all(HISTORY)

# Print summary table
if HISTORY.epochs:
    print("\n" + "="*80)
    print("TRAINING SUMMARY")
    print("="*80)
    print(f"{'Epoch':>6} {'Stage':<25} {'Loss':>8} {'Overall':>8} {'Minority':>8} "
          f"{'GENE':>8} {'DISEASE':>8} {'CHEM':>8}")
    print("-"*80)
    for i, ep in enumerate(HISTORY.epochs):
        print(f"{ep:>6d} {HISTORY.stages[i]:<25} {HISTORY.train_loss[i]:>8.4f} "
              f"{HISTORY.overall_f1[i]:>8.4f} {HISTORY.minority_f1[i]:>8.4f} "
              f"{HISTORY.gene_f1[i]:>8.4f} {HISTORY.disease_f1[i]:>8.4f} "
              f"{HISTORY.chemical_f1[i]:>8.4f}")
    print("="*80)